# Notebook 03 - Hydrodynamic Drag

## Purpose

This notebook introduces the first **velocity-dependent resistive force** into HydroLab-3D.

The previous notebooks established a validated translational lineage:

- Notebook 00: position propagation from velocity;
- Notebook 01: body/world reference-frame transformations;
- Notebook 02: force-driven acceleration and velocity evolution.

Until now, the translational dynamics have been idealized. Under a constant applied force, nothing opposes the vehicle's motion, so its speed can continue increasing indefinitely.

Notebook 03 removes that idealization by introducing **hydrodynamic drag**.

The central question is:

> **Given that a moving underwater vehicle experiences velocity-dependent resistance, how does hydrodynamic drag modify its net force, acceleration, velocity evolution, and resulting trajectory?**

The model will deliberately remain small:

```text
Current Velocity
      ↓
Hydrodynamic Drag
      ↓
Applied Force + Drag
      ↓
Net Force
      ↓
Acceleration
      ↓
Velocity Evolution
      ↓
Position Evolution
```

The objective is not yet to construct a complete marine vehicle model.

The objective is to establish a mathematically trustworthy and reusable **velocity-dependent resistance layer** that later HydroLab-3D notebooks can inherit.

---

## Why This Matters

A real underwater vehicle cannot generally accelerate forever under constant thrust.

As vehicle speed increases relative to the surrounding water, hydrodynamic resistance also increases. Eventually, under a constant applied force, the resisting drag may grow sufficiently large that:

$$
\mathbf{F}_{applied}
+
\mathbf{F}_D
\approx
\mathbf{0}
$$

At that condition:

$$
\mathbf{F}_{net}
\approx
\mathbf{0}
$$

and therefore:

$$
\mathbf{a}
\approx
\mathbf{0}
$$

The vehicle then approaches a **terminal velocity** rather than continuing to accelerate indefinitely.

This behavior is foundational for meaningful marine-vehicle dynamics.

Without drag, later models involving:

- buoyancy;
- gravity;
- thrusters;
- currents;
- closed-loop control;
- trajectory tracking;
- energy consumption;
- and full 6-DOF dynamics

would operate on an unrealistically unconstrained translational model.

Notebook 03 therefore introduces the first mechanism through which the surrounding fluid directly modifies vehicle motion.

---

## What We Will Implement

Notebook 03 will implement and validate:

- an effective quadratic translational drag model;
- drag computed from the current 3D velocity vector;
- net-force construction from applied force and drag;
- drag-driven deceleration with no applied force;
- constant applied force combined with quadratic resistance;
- numerical convergence toward an analytically predicted terminal speed;
- integration through the already validated HydroLab-3D translational dynamics functions;
- finite-value and deterministic numerical behavior;
- limited timestep-sensitivity analysis only if required by the drag validation;
- promotion of the validated drag implementation into the reusable source package;
- focused automated regression tests for the production drag function;
- persistent scientific artifacts containing the final validated Notebook 03 experiment.

The foundational validation will use **still water** and an aligned translational frame.

Changing orientation, ocean currents, rotational hydrodynamics, and full 6-DOF effects are intentionally excluded.

---

## What We Inherit From Notebook 00

Notebook 00 validated deterministic 3D translational position propagation using:

$$
\mathbf{p}_{k+1}
=
\mathbf{p}_k
+
\mathbf{v}_k\Delta t
$$

for piecewise-constant velocity over one timestep.

That notebook validated stationary motion, independent and simultaneous Cartesian motion, piecewise-constant velocity commands, analytical agreement, deterministic reproducibility, finite-state behavior, and persistent trajectory generation.

Its reusable position integrator was subsequently promoted into:

`src/hydrolab3d/dynamics/kinematics.py`

as:

`integrate_position()`

Notebook 03 therefore does **not** need to redesign position integration.

Once drag determines the current acceleration and velocity evolution, the already validated production position integrator remains responsible for propagating translational position.

---

## What We Inherit From Notebook 01

Notebook 01 established the HydroLab-3D body/world reference-frame contract.

For a body-frame velocity:

$$
\mathbf{v}_B
$$

the corresponding world-frame velocity is:

$$
\mathbf{v}_W
=
R_B^W
\mathbf{v}_B
$$

with:

$$
R_B^W
=
R_z(\psi)
R_y(\theta)
R_x(\phi)
$$

under the established convention:

**World frame**

- $+x_W$: horizontal x;
- $+y_W$: horizontal y;
- $+z_W$: upward.

**Body frame**

- $+x_B$: forward;
- $+y_B$: left;
- $+z_B$: upward.

Positive rotations follow the right-hand rule, and orientation angles are represented internally in radians.

The validated rotation helpers were promoted into:

`src/hydrolab3d/utils/transforms.py`

including:

- `rotation_x()`;
- `rotation_y()`;
- `rotation_z()`;
- `rotation_body_to_world()`.

This reference-frame infrastructure remains available to Notebook 03.

However, the purpose of the present notebook is to validate hydrodynamic resistance rather than repeat orientation experiments. The primary drag validation will therefore remain in one aligned frame unless a fixed-orientation compatibility check is later shown to provide unique engineering evidence.

---

## What We Inherit From Notebook 02

Notebook 02 is the immediate physical dependency of Notebook 03.

It validated Newtonian translational dynamics:

$$
\mathbf{F}
=
m\mathbf{a}
$$

therefore:

$$
\mathbf{a}
=
\frac{\mathbf{F}}{m}
$$

and the velocity update:

$$
\mathbf{v}_{k+1}
=
\mathbf{v}_k
+
\mathbf{a}_k\Delta t
$$

while retaining the Notebook 00 position update:

$$
\mathbf{p}_{k+1}
=
\mathbf{p}_k
+
\mathbf{v}_k\Delta t
$$

Notebook 02 explicitly selected **explicit Euler integration**.

Its discrete update convention is:

1. evaluate the force acting at the beginning of the timestep;
2. compute acceleration from that force;
3. update position using the beginning-of-step velocity;
4. update velocity using the current acceleration.

The reusable dynamics functions were promoted into:

`src/hydrolab3d/dynamics/rigid_body.py`

including:

- `acceleration_from_force()`;
- `integrate_velocity()`.

The previously validated position integrator remains in:

`src/hydrolab3d/dynamics/kinematics.py`

Notebook 02 also populated the dynamics package interface and introduced automated regression coverage in:

`tests/test_dynamics.py`

with the completed dynamics test suite reporting:

`16 passed`

Its canonical constant-force validation artifacts were stored separately under:

`results/notebooks/02_basic_dynamics/`

including:

- `data/constant_force_validation.csv`;
- `plots/velocity_validation.png`;
- `plots/position_validation.png`.

Notebook 03 will not overwrite or recreate those results.

The force-to-acceleration, velocity-integration, and position-integration layers are already validated dependencies.

The new responsibility of Notebook 03 is only to determine the additional force contributed by hydrodynamic resistance and feed that force into the existing dynamics chain.

---

## Mathematical Model

Let the translational vehicle velocity relative to still water be:

$$
\mathbf{v}
$$

The vehicle speed is:

$$
\|\mathbf{v}\|
$$

Notebook 03 will use the reduced-order quadratic drag model:

$$
\mathbf{F}_D
=
-C_D
\|\mathbf{v}\|
\mathbf{v}
$$

where:

- $\mathbf{F}_D$ is the hydrodynamic drag-force vector;
- $\mathbf{v}$ is the translational velocity used to evaluate the drag model;
- $\|\mathbf{v}\|$ is the scalar vehicle speed;
- $C_D$ is an **effective quadratic drag coefficient** for this reduced-order model.

The coefficient $C_D$ used here is **not being treated as the conventional dimensionless hydrodynamic drag coefficient** appearing in:

$$
F_D
=
\frac{1}{2}
\rho
C_D
A
v^2
$$

Notebook 03 does not yet introduce fluid density, projected reference area, geometry-dependent coefficients, or directional hydrodynamic parameter identification.

Instead, the present $C_D$ absorbs the required proportionality into one effective parameter so that the notebook can isolate and validate the essential behavior:

$$
\text{higher speed}
\rightarrow
\text{stronger resistance}
$$

The translational net force becomes:

$$
\mathbf{F}_{net}
=
\mathbf{F}_{applied}
+
\mathbf{F}_D
$$

therefore:

$$
m\dot{\mathbf{v}}
=
\mathbf{F}_{applied}
-
C_D
\|\mathbf{v}\|
\mathbf{v}
$$

The resulting architecture is:

```text
Current Velocity
      ↓
Quadratic Drag Model
      ↓
Drag Force
      ↓
Applied Force + Drag Force
      ↓
Net Force
      ↓
acceleration_from_force(...)
      ↓
Acceleration
      ↓
integrate_position(...)
integrate_velocity(...)
```

### Terminal Speed Under Constant One-Axis Force

For positive one-axis motion with constant applied force:

$$
F_A > 0
$$

and quadratic resistance:

$$
F_D
=
-C_D|v|v
$$

terminal motion occurs when acceleration tends toward zero:

$$
F_A
+
F_D
=
0
$$

For positive terminal velocity:

$$
F_A
=
C_Dv_\infty^2
$$

therefore:

$$
v_\infty
=
\sqrt{\frac{F_A}{C_D}}
$$

This relation provides the primary analytical reference for Notebook 03.

The numerical simulation should **approach** this predicted terminal speed.

Exact equality at a finite simulation time will not be assumed.

---

## Numerical Integration Convention

Notebook 03 retains the explicit-Euler convention validated in Notebook 02.

At timestep $k$, drag is evaluated from the **current beginning-of-step velocity**:

$$
\mathbf{F}_{D,k}
=
-C_D
\|\mathbf{v}_k\|
\mathbf{v}_k
$$

The net force is then:

$$
\mathbf{F}_{net,k}
=
\mathbf{F}_{applied,k}
+
\mathbf{F}_{D,k}
$$

Acceleration follows from:

$$
\mathbf{a}_k
=
\frac{\mathbf{F}_{net,k}}{m}
$$

Position is updated using the beginning-of-step velocity:

$$
\mathbf{p}_{k+1}
=
\mathbf{p}_k
+
\mathbf{v}_k\Delta t
$$

Velocity is then updated using:

$$
\mathbf{v}_{k+1}
=
\mathbf{v}_k
+
\mathbf{a}_k\Delta t
$$

Therefore the complete timestep ordering is:

```text
v_k
 ↓
evaluate drag
 ↓
construct net force
 ↓
compute acceleration
 ↓
update position using v_k
 ↓
update velocity using a_k
 ↓
state k+1
```

Drag will not be evaluated using the newly updated velocity while other quantities still use the beginning-of-step state.

The discrete-time convention must remain internally consistent.

Unlike Notebook 02's constant-acceleration case, the acceleration here changes with velocity. Explicit Euler therefore introduces genuine numerical approximation into the velocity evolution as well as the position evolution.

Numerical behavior must consequently be distinguished from the underlying physical model.

---

## Expected Result

If the implementation is physically and numerically correct, the following behavior should emerge.

### Zero Velocity

For:

$$
\mathbf{v}
=
\mathbf{0}
$$

drag should be:

$$
\mathbf{F}_D
=
\mathbf{0}
$$

### Drag Opposes Motion

For every nonzero velocity:

$$
\mathbf{F}_D
\cdot
\mathbf{v}
<
0
$$

This means the drag force removes mechanical energy rather than injecting it.

At zero velocity:

$$
\mathbf{F}_D
\cdot
\mathbf{v}
=
0
$$

### Velocity Reversal Reverses Drag

If:

$$
\mathbf{v}
\rightarrow
-\mathbf{v}
$$

then:

$$
\mathbf{F}_D(-\mathbf{v})
=
-\mathbf{F}_D(\mathbf{v})
$$

### Higher Speed Produces Stronger Drag

For motion in an unchanged direction, doubling velocity magnitude should produce four times the drag magnitude:

$$
\mathbf{v}_2
=
2\mathbf{v}_1
$$

therefore:

$$
\|\mathbf{F}_{D,2}\|
=
4
\|\mathbf{F}_{D,1}\|
$$

### Passive Motion Decelerates

With:

$$
\mathbf{F}_{applied}
=
\mathbf{0}
$$

a moving vehicle should lose speed monotonically under drag, provided the numerical timestep is suitable.

The drag model must not spontaneously increase kinetic energy.

### Constant Applied Force Produces Terminal Motion

Under a constant applied force, acceleration should initially be positive when drag is smaller than the applied force.

As speed increases:

$$
\|\mathbf{F}_D\|
\uparrow
$$

so:

$$
\|\mathbf{F}_{net}\|
\downarrow
$$

and consequently:

$$
\|\mathbf{a}\|
\downarrow
$$

The velocity should asymptotically approach the analytical terminal speed:

$$
v_\infty
=
\sqrt{\frac{F_A}{C_D}}
$$

Near terminal motion:

$$
\mathbf{F}_{applied}
+
\mathbf{F}_D
\approx
\mathbf{0}
$$

and:

$$
\mathbf{a}
\approx
\mathbf{0}
$$

---

## Validation Criteria

Notebook 03 will not treat visually plausible damping as sufficient validation.

The drag layer must be supported by quantitative tests.

The validation program will require evidence that:

- zero velocity produces zero drag;
- zero drag coefficient produces zero drag;
- nonzero drag acts opposite the velocity vector;
- reversing velocity reverses drag;
- drag magnitude follows the expected quadratic speed scaling;
- arbitrary 3D velocity produces drag anti-parallel to that velocity;
- passive motion does not gain kinetic energy;
- passive speed decreases as expected under drag;
- constant applied force causes velocity to approach the analytically predicted terminal speed;
- the terminal-speed comparison is measured using an explicit numerical error metric;
- applied force and drag approach force balance near terminal motion;
- simulated quantities remain finite;
- repeated deterministic runs reproduce the same result;
- the selected timestep does not introduce nonphysical sign flipping, overshoot, or instability;
- any timestep-sensitivity study is performed only if required to separate integration error from drag-model behavior.

No arbitrary terminal-velocity tolerance will be declared in advance.

Any numerical threshold used to determine convergence must be justified by the simulation duration, timestep, expected asymptotic behavior, and observed numerical accuracy.

---

## Relationship to HydroLab-3D

Notebook 03 occupies the next layer in the progressive HydroLab-3D dynamics lineage:

```text
Notebook 00
Position Propagation
      ↓
Notebook 01
Orientation and Reference Frames
      ↓
Notebook 02
Force → Acceleration → Velocity
      ↓
Notebook 03
Velocity-Dependent Hydrodynamic Drag
      ↓
Notebook 04
Gravity and Buoyancy
```

The development philosophy remains:

```text
Notebook Prototype
      ↓
Mathematical / Numerical Validation
      ↓
Promote Reusable Implementation into src/
      ↓
Add Automated Regression Tests
      ↓
Notebook Uses Production Implementation
      ↓
Next Notebook Inherits Validated Capability
```

Notebook 03 therefore represents more than an isolated numerical experiment.

Its successful result should become the reusable resistance component upon which later marine dynamics are assembled.

---

## Scope Boundary

Notebook 03 deliberately does **not** implement:

- gravity;
- buoyancy;
- thruster models;
- actuator saturation or actuator dynamics;
- ocean currents;
- current-relative flow;
- linear-plus-quadratic damping models;
- anisotropic vehicle drag coefficients;
- geometry-dependent drag parameterization;
- computational fluid dynamics;
- turbulence;
- added mass;
- Coriolis or centripetal effects;
- rotational drag;
- rotational rigid-body dynamics;
- full 6-DOF UUV dynamics;
- sensors;
- controllers;
- autonomous navigation;
- ROS 2;
- reinforcement learning.

The vehicle is still represented by a reduced translational model.

The surrounding water is stationary.

The purpose of this notebook is exactly one new physical layer:

$$
\boxed{
\text{velocity}
\rightarrow
\text{hydrodynamic resistance}
}
$$

Everything else remains outside scope until later notebooks.

---

## Expected Outputs

Persistent artifacts will be created only when they provide useful scientific evidence.

Tentative outputs may include:

### Numerical Data

A canonical CSV containing quantities such as:

- simulation time;
- applied force;
- drag force;
- net force;
- acceleration;
- velocity;
- vehicle speed;
- position;
- analytically predicted terminal speed;
- terminal-speed error.

### Validation Figures

Potential figures include:

- velocity versus time with the analytical terminal-speed reference;
- applied, drag, and/or net force versus time to demonstrate force balance;
- passive deceleration versus time if that experiment provides distinct validation evidence.

These filenames and artifacts remain tentative until the corresponding experiments have been validated.

No plot will be preserved merely for presentation.

Every persistent artifact should document a physical or numerical result.

---

## Output Directory

All Notebook 03 persistent outputs will belong under:

`results/notebooks/03_hydrodynamic_drag/`

Only subdirectories actually required by the validated experiments will be created.

Notebook 03 will not modify the persistent artifacts produced by Notebooks 00, 01, or 02.

---

## Engineering Handoff Target

The successful handoff from Notebook 03 should allow subsequent HydroLab-3D development to assume that translational hydrodynamic resistance has already been validated.

The reusable translational force balance should support:

$$
m\dot{\mathbf{v}}
=
\mathbf{F}_{applied}
+
\mathbf{F}_D
$$

with:

$$
\mathbf{F}_D
=
-C_D
\|\mathbf{v}\|
\mathbf{v}
$$

Notebook 04 should therefore be able to extend the force balance with gravity and buoyancy rather than reopening the drag problem.

Conceptually, Notebook 04 should be able to proceed from:

$$
m\dot{\mathbf{v}}
=
\mathbf{F}_{applied}
+
\mathbf{F}_D
$$

toward a model containing additional forces such as:

$$
\mathbf{F}_G
$$

and:

$$
\mathbf{F}_B
$$

while inheriting the following as already established:

- position integration;
- velocity integration;
- Newtonian force-to-acceleration dynamics;
- the explicit-Euler timestep convention;
- body/world reference-frame infrastructure;
- quadratic translational drag behavior;
- drag direction and energy-dissipation properties;
- passive drag-induced deceleration;
- constant-force terminal-velocity behavior;
- the production drag API and its regression tests, once Notebook 03 validation is complete.

The intended lineage is therefore:

$$
\boxed{
\text{trustworthy mathematics}
\rightarrow
\text{reproducible validation}
\rightarrow
\text{reusable production physics}
\rightarrow
\text{progressively richer marine dynamics}
}
$$

---

## Step 1 - Audit Inherited Production Code and Drag Development Targets

Before implementing hydrodynamic drag, we must establish the **actual HydroLab-3D repository state** inherited from the completed notebooks.

The attached completed notebooks confirm that Notebook 03 is not beginning from an empty dynamics stack.

Notebook 02 ultimately promoted and used:

- `integrate_position()` from `src/hydrolab3d/dynamics/kinematics.py`;
- `acceleration_from_force()` from `src/hydrolab3d/dynamics/rigid_body.py`;
- `integrate_velocity()` from `src/hydrolab3d/dynamics/rigid_body.py`;
- the Notebook 01 rotation utilities from `src/hydrolab3d/utils/transforms.py`.

Notebook 02 also established automated dynamics regression tests and generated persistent validation artifacts. Its explicit-Euler convention is therefore an inherited numerical contract rather than something Notebook 03 should redesign.

Notebook 03 introduces only one fundamentally new physical mapping:

$$
\mathbf{v}
\rightarrow
\mathbf{F}_D
$$

with the intended reduced-order model:

$$
\mathbf{F}_D
=
-C_D
\|\mathbf{v}\|
\mathbf{v}
$$

However, HydroLab-3D's tentative repository structure already includes candidate files such as:

`src/hydrolab3d/dynamics/drag.py`

and:

`tests/test_drag.py`

Their existence does **not** imply that drag has already been implemented.

They may be:

- absent;
- empty placeholders;
- partially populated;
- or contain code that must be preserved.

The same principle applies to the inherited production modules. Before Notebook 03 imports or modifies anything, we will inspect the repository directly.

### Purpose of This Step

This is a **read-only repository audit**.

We will inspect:

- `src/hydrolab3d/dynamics/kinematics.py`;
- `src/hydrolab3d/dynamics/rigid_body.py`;
- `src/hydrolab3d/dynamics/drag.py`;
- `src/hydrolab3d/dynamics/__init__.py`;
- `src/hydrolab3d/utils/transforms.py`;
- `tests/test_dynamics.py`;
- `tests/test_drag.py`.

We will also inspect the existing notebook-result directories so that Notebook 03 does not overwrite earlier artifacts and so that its own output directory begins from a known state.

### What This Step Must Establish

Before new drag code is written, we need to know:

1. whether the validated Notebook 00 position integrator is present in production;
2. whether the validated Notebook 02 force and velocity functions are present in production;
3. whether the validated Notebook 01 transformation utilities are present in production;
4. whether `drag.py` is absent, empty, or already populated;
5. whether `test_drag.py` is absent, empty, or already populated;
6. what automated dynamics tests currently exist;
7. what persistent result directories currently exist;
8. whether `results/notebooks/03_hydrodynamic_drag/` already contains anything that must be preserved.

No source file, test file, or result artifact will be modified during this step.

In [1]:
from pathlib import Path


# ---------------------------------------------------------------------
# Locate HydroLab-3D repository root
# ---------------------------------------------------------------------

repo_root = Path.cwd().resolve()

if repo_root.name == "notebooks":
    repo_root = repo_root.parent

print("HydroLab-3D repository root:")
print(repo_root)
print()


# ---------------------------------------------------------------------
# Files required by Notebook 03
# ---------------------------------------------------------------------

files_to_inspect = [
    repo_root / "src" / "hydrolab3d" / "dynamics" / "kinematics.py",
    repo_root / "src" / "hydrolab3d" / "dynamics" / "rigid_body.py",
    repo_root / "src" / "hydrolab3d" / "dynamics" / "drag.py",
    repo_root / "src" / "hydrolab3d" / "dynamics" / "__init__.py",
    repo_root / "src" / "hydrolab3d" / "utils" / "transforms.py",
    repo_root / "tests" / "test_dynamics.py",
    repo_root / "tests" / "test_drag.py",
]


print("=" * 80)
print("SOURCE / TEST FILE AUDIT")
print("=" * 80)
print()


for path in files_to_inspect:
    try:
        display_path = path.relative_to(repo_root)
    except ValueError:
        display_path = path

    print("=" * 80)
    print(display_path)
    print("=" * 80)

    if not path.exists():
        print("[FILE DOES NOT EXIST]")
        print()
        continue

    contents = path.read_text(encoding="utf-8")

    if not contents.strip():
        print("[FILE IS EMPTY]")
    else:
        print(contents)

    print()


# ---------------------------------------------------------------------
# Persistent notebook-result audit
# ---------------------------------------------------------------------

results_root = (
    repo_root
    / "results"
    / "notebooks"
)

notebook_result_dirs = [
    results_root / "00_basic_3d_kinematics",
    results_root / "01_orientation",
    results_root / "02_basic_dynamics",
    results_root / "03_hydrodynamic_drag",
]


print("=" * 80)
print("NOTEBOOK RESULT DIRECTORY AUDIT")
print("=" * 80)
print()


for directory in notebook_result_dirs:
    try:
        display_path = directory.relative_to(repo_root)
    except ValueError:
        display_path = directory

    print("-" * 80)
    print(display_path)

    if not directory.exists():
        print("[DIRECTORY DOES NOT EXIST]")
        continue

    entries = sorted(
        path.relative_to(directory)
        for path in directory.rglob("*")
    )

    if not entries:
        print("[DIRECTORY IS EMPTY]")
        continue

    for entry in entries:
        full_path = directory / entry

        if full_path.is_dir():
            print(f"[DIR ] {entry}")
        else:
            print(f"[FILE] {entry}")

print()
print("Repository audit complete. No files were modified.")

HydroLab-3D repository root:
/home/agniacolyte/HydroLab-3D

SOURCE / TEST FILE AUDIT

src/hydrolab3d/dynamics/kinematics.py
"""Basic translational kinematics for HydroLab-3D."""

import numpy as np


def integrate_position(position, velocity, dt):
    """
    Advance a 3D position by one constant-velocity timestep.

    Parameters
    ----------
    position : np.ndarray
        Current world-frame position [x, y, z] in meters.
    velocity : np.ndarray
        Current world-frame velocity [vx, vy, vz] in meters/second.
    dt : float
        Simulation timestep in seconds.

    Returns
    -------
    np.ndarray
        Updated world-frame position after one timestep.
    """
    position = np.asarray(position, dtype=float)
    velocity = np.asarray(velocity, dtype=float)

    if position.shape != (3,):
        raise ValueError("position must have shape (3,)")

    if velocity.shape != (3,):
        raise ValueError("velocity must have shape (3,)")

    if not np.all(np.isfinite(posit

## Step 2 - Implement and Validate the Notebook-Local Quadratic Drag Function

The repository audit confirmed that:

- the validated production position integrator exists;
- the validated production force-to-acceleration and velocity integrators exist;
- the validated body/world transformation utilities exist;
- `src/hydrolab3d/dynamics/drag.py` is currently empty;
- `tests/test_drag.py` is currently empty.

Therefore the next task is to implement the new drag model **locally inside this notebook first**.

We will not promote anything into `src/` yet.

The reduced-order quadratic drag model is:

$$
\mathbf{F}_D
=
-C_D
\|\mathbf{v}\|
\mathbf{v}
$$

where:

- $\mathbf{v}$ is a 3D translational velocity vector;
- $\|\mathbf{v}\|$ is its Euclidean magnitude;
- $C_D$ is a nonnegative effective quadratic drag coefficient;
- $\mathbf{F}_D$ is the resulting drag-force vector.

The effective coefficient $C_D$ is not being treated as the conventional dimensionless coefficient from a full hydrodynamic expression involving density and reference area. It is a reduced-model coefficient that directly maps velocity into drag force.

### Required Input Contract

The notebook-local implementation will enforce:

- `velocity` must have shape `(3,)`;
- all velocity components must be finite;
- `drag_coefficient` must be a scalar;
- `drag_coefficient` must be finite;
- `drag_coefficient >= 0`.

A negative coefficient is invalid because:

$$
C_D < 0
$$

would cause:

$$
\mathbf{F}_D
\cdot
\mathbf{v}
>
0
$$

for nonzero velocity, meaning the supposed drag term would inject kinetic energy rather than dissipate it.

### Initial Validation Targets

This step will validate the algebraic drag function itself before any time integration is introduced.

We will check:

1. zero velocity produces zero drag;
2. zero drag coefficient produces zero drag;
3. a known one-axis case produces the expected force;
4. drag opposes nonzero motion;
5. reversing velocity reverses drag;
6. doubling speed in the same direction multiplies drag magnitude by four;
7. an arbitrary 3D velocity produces drag anti-parallel to velocity;
8. invalid shapes, nonfinite inputs, and invalid coefficients are rejected.

These are direct properties of the model itself.

No trajectory simulation is required yet.

If all checks pass, the drag-force mapping can be considered locally validated at the algebraic level, and the next step can connect it to the inherited Notebook 02 dynamics chain.

In [2]:
import numpy as np


def quadratic_drag_force(velocity, drag_coefficient):
    """
    Compute reduced-order quadratic translational drag.

    Parameters
    ----------
    velocity : array-like
        3D translational velocity [vx, vy, vz] in m/s.
    drag_coefficient : float
        Nonnegative effective quadratic drag coefficient.

    Returns
    -------
    np.ndarray
        Drag-force vector [Fx, Fy, Fz] in newtons.
    """
    velocity = np.asarray(velocity, dtype=float)

    if velocity.shape != (3,):
        raise ValueError("velocity must have shape (3,)")

    if not np.all(np.isfinite(velocity)):
        raise ValueError("velocity must contain only finite values")

    if not np.isscalar(drag_coefficient):
        raise ValueError("drag_coefficient must be a scalar")

    drag_coefficient = float(drag_coefficient)

    if not np.isfinite(drag_coefficient):
        raise ValueError("drag_coefficient must be finite")

    if drag_coefficient < 0.0:
        raise ValueError("drag_coefficient must be nonnegative")

    speed = np.linalg.norm(velocity)

    return -drag_coefficient * speed * velocity


# ---------------------------------------------------------------------
# 1. Zero velocity -> zero drag
# ---------------------------------------------------------------------

drag_zero_velocity = quadratic_drag_force(
    np.zeros(3),
    2.5,
)

assert np.array_equal(
    drag_zero_velocity,
    np.zeros(3),
)


# ---------------------------------------------------------------------
# 2. Zero coefficient -> zero drag
# ---------------------------------------------------------------------

drag_zero_coefficient = quadratic_drag_force(
    np.array([1.0, -2.0, 0.5]),
    0.0,
)

assert np.array_equal(
    drag_zero_coefficient,
    np.zeros(3),
)


# ---------------------------------------------------------------------
# 3. Known one-axis case
#
# v = [3, 0, 0] m/s
# ||v|| = 3 m/s
# C_D = 2
#
# F_D = -2 * 3 * [3, 0, 0]
#     = [-18, 0, 0] N
# ---------------------------------------------------------------------

velocity_known = np.array([3.0, 0.0, 0.0])
coefficient_known = 2.0

drag_known = quadratic_drag_force(
    velocity_known,
    coefficient_known,
)

expected_known = np.array([-18.0, 0.0, 0.0])

assert np.allclose(
    drag_known,
    expected_known,
    atol=1e-12,
    rtol=0.0,
)


# ---------------------------------------------------------------------
# 4. Drag opposes motion
# ---------------------------------------------------------------------

velocity_general = np.array([2.0, -1.0, 0.5])

drag_general = quadratic_drag_force(
    velocity_general,
    1.7,
)

power_like_term = np.dot(
    drag_general,
    velocity_general,
)

assert power_like_term < 0.0


# ---------------------------------------------------------------------
# 5. Odd symmetry:
#
# F_D(-v) = -F_D(v)
# ---------------------------------------------------------------------

drag_reversed = quadratic_drag_force(
    -velocity_general,
    1.7,
)

assert np.allclose(
    drag_reversed,
    -drag_general,
    atol=1e-12,
    rtol=0.0,
)


# ---------------------------------------------------------------------
# 6. Quadratic speed scaling:
#
# F_D(2v) should have 4x the magnitude of F_D(v)
# ---------------------------------------------------------------------

drag_base = quadratic_drag_force(
    velocity_general,
    1.7,
)

drag_double_speed = quadratic_drag_force(
    2.0 * velocity_general,
    1.7,
)

drag_scaling_ratio = (
    np.linalg.norm(drag_double_speed)
    / np.linalg.norm(drag_base)
)

assert np.isclose(
    drag_scaling_ratio,
    4.0,
    atol=1e-12,
    rtol=0.0,
)


# ---------------------------------------------------------------------
# 7. Arbitrary 3D velocity -> drag anti-parallel to velocity
#
# For nonzero v:
#
# F_D / ||F_D|| = -v / ||v||
# ---------------------------------------------------------------------

velocity_3d = np.array([1.25, -2.5, 0.75])

drag_3d = quadratic_drag_force(
    velocity_3d,
    0.8,
)

velocity_direction = (
    velocity_3d
    / np.linalg.norm(velocity_3d)
)

drag_direction = (
    drag_3d
    / np.linalg.norm(drag_3d)
)

assert np.allclose(
    drag_direction,
    -velocity_direction,
    atol=1e-12,
    rtol=0.0,
)


# ---------------------------------------------------------------------
# 8. Invalid-input validation
# ---------------------------------------------------------------------

invalid_cases_passed = 0


def expect_value_error(func):
    global invalid_cases_passed

    try:
        func()
    except ValueError:
        invalid_cases_passed += 1
    else:
        raise AssertionError("Expected ValueError was not raised")


expect_value_error(
    lambda: quadratic_drag_force(
        np.ones(2),
        1.0,
    )
)

expect_value_error(
    lambda: quadratic_drag_force(
        np.array([1.0, np.nan, 0.0]),
        1.0,
    )
)

expect_value_error(
    lambda: quadratic_drag_force(
        np.array([1.0, np.inf, 0.0]),
        1.0,
    )
)

expect_value_error(
    lambda: quadratic_drag_force(
        np.ones(3),
        [1.0],
    )
)

expect_value_error(
    lambda: quadratic_drag_force(
        np.ones(3),
        np.nan,
    )
)

expect_value_error(
    lambda: quadratic_drag_force(
        np.ones(3),
        np.inf,
    )
)

expect_value_error(
    lambda: quadratic_drag_force(
        np.ones(3),
        -1.0,
    )
)


# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------

print("Quadratic drag algebraic validation")
print("-" * 50)
print(f"Zero-velocity drag:          {drag_zero_velocity}")
print(f"Zero-coefficient drag:       {drag_zero_coefficient}")
print(f"Known-case drag:             {drag_known}")
print(f"General 3D drag:             {drag_general}")
print(f"F_D dot v:                   {power_like_term:.12f}")
print(f"Quadratic scaling ratio:     {drag_scaling_ratio:.12f}")
print(f"Invalid cases rejected:      {invalid_cases_passed}/7")
print()
print("All algebraic drag checks passed.")

Quadratic drag algebraic validation
--------------------------------------------------
Zero-velocity drag:          [-0. -0. -0.]
Zero-coefficient drag:       [-0.  0. -0.]
Known-case drag:             [-18.  -0.  -0.]
General 3D drag:             [-7.79037868  3.89518934 -1.94759467]
F_D dot v:                   -20.449744038740
Quadratic scaling ratio:     4.000000000000
Invalid cases rejected:      7/7

All algebraic drag checks passed.


## Step 3 - Connect Drag to the Inherited Dynamics Chain and Validate Passive Deceleration

The drag-force mapping has now passed its algebraic validation.

The next task is to connect that force to the already validated HydroLab-3D translational dynamics pipeline.

For this experiment, no external force is applied:

$$
\mathbf{F}_{applied}
=
\mathbf{0}
$$

Therefore the net force is entirely hydrodynamic drag:

$$
\mathbf{F}_{net}
=
\mathbf{F}_D
$$

with:

$$
\mathbf{F}_D
=
-C_D
\|\mathbf{v}\|
\mathbf{v}
$$

Acceleration is computed using the inherited production function:

$$
\mathbf{a}
=
\frac{\mathbf{F}_{net}}{m}
$$

and the validated explicit-Euler update convention remains:

$$
\mathbf{p}_{k+1}
=
\mathbf{p}_k
+
\mathbf{v}_k \Delta t
$$

followed by:

$$
\mathbf{v}_{k+1}
=
\mathbf{v}_k
+
\mathbf{a}_k \Delta t
$$

### Physical Expectation

Because drag always opposes motion:

$$
\mathbf{F}_D
\cdot
\mathbf{v}
<
0
$$

the vehicle should lose kinetic energy.

For a vehicle moving in one fixed direction with no other applied force, its speed should decrease monotonically.

The kinetic energy is:

$$
E_k
=
\frac{1}{2}
m
\|\mathbf{v}\|^2
$$

so we expect:

$$
E_{k,k+1}
\le
E_{k}
$$

throughout a numerically stable passive-drag simulation.

The vehicle should continue moving forward while gradually slowing, rather than stopping instantaneously.

### Purpose of This Validation

This step tests something that the algebraic checks could not:

> Does the new drag model behave correctly when inserted into the actual inherited HydroLab-3D time-integration chain?

We will verify:

- imported production dynamics functions work with the notebook-local drag model;
- speed decreases monotonically;
- kinetic energy decreases monotonically;
- drag remains dissipative;
- no velocity sign reversal occurs;
- no NaN or infinite values appear;
- the vehicle continues translating while gradually decelerating.

No terminal-velocity experiment is introduced yet.

This step isolates **passive drag-induced slowdown** before constant applied force is added.

In [3]:
import sys
from pathlib import Path
import numpy as np


# ---------------------------------------------------------------------
# Make production package importable
# ---------------------------------------------------------------------

repo_root = Path.cwd().resolve()

if repo_root.name == "notebooks":
    repo_root = repo_root.parent

src_path = repo_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))


from hydrolab3d.dynamics import (
    acceleration_from_force,
    integrate_position,
    integrate_velocity,
)


# ---------------------------------------------------------------------
# Passive-drag simulation parameters
# ---------------------------------------------------------------------

mass = 20.0                  # kg
drag_coefficient = 3.0       # effective reduced-order coefficient
dt = 0.01                    # s
simulation_time = 8.0        # s

initial_position = np.array([0.0, 0.0, -10.0])
initial_velocity = np.array([2.0, 0.0, 0.0])

num_steps = int(round(simulation_time / dt))


# ---------------------------------------------------------------------
# Allocate histories
# ---------------------------------------------------------------------

time = np.linspace(
    0.0,
    simulation_time,
    num_steps + 1,
)

positions = np.zeros((num_steps + 1, 3))
velocities = np.zeros((num_steps + 1, 3))
drag_forces = np.zeros((num_steps + 1, 3))
accelerations = np.zeros((num_steps + 1, 3))
speeds = np.zeros(num_steps + 1)
kinetic_energy = np.zeros(num_steps + 1)


positions[0] = initial_position
velocities[0] = initial_velocity
speeds[0] = np.linalg.norm(initial_velocity)
kinetic_energy[0] = 0.5 * mass * speeds[0] ** 2


# ---------------------------------------------------------------------
# Explicit-Euler simulation
# ---------------------------------------------------------------------

for k in range(num_steps):

    velocity_k = velocities[k]
    position_k = positions[k]

    drag_k = quadratic_drag_force(
        velocity_k,
        drag_coefficient,
    )

    acceleration_k = acceleration_from_force(
        drag_k,
        mass,
    )

    position_next = integrate_position(
        position_k,
        velocity_k,
        dt,
    )

    velocity_next = integrate_velocity(
        velocity_k,
        acceleration_k,
        dt,
    )

    drag_forces[k] = drag_k
    accelerations[k] = acceleration_k

    positions[k + 1] = position_next
    velocities[k + 1] = velocity_next

    speeds[k + 1] = np.linalg.norm(
        velocity_next
    )

    kinetic_energy[k + 1] = (
        0.5
        * mass
        * speeds[k + 1] ** 2
    )


# Evaluate final-state drag and acceleration for complete histories
drag_forces[-1] = quadratic_drag_force(
    velocities[-1],
    drag_coefficient,
)

accelerations[-1] = acceleration_from_force(
    drag_forces[-1],
    mass,
)


# ---------------------------------------------------------------------
# Validation metrics
# ---------------------------------------------------------------------

speed_differences = np.diff(speeds)

energy_differences = np.diff(
    kinetic_energy
)

drag_power = np.einsum(
    "ij,ij->i",
    drag_forces,
    velocities,
)

all_finite = (
    np.all(np.isfinite(positions))
    and np.all(np.isfinite(velocities))
    and np.all(np.isfinite(drag_forces))
    and np.all(np.isfinite(accelerations))
    and np.all(np.isfinite(speeds))
    and np.all(np.isfinite(kinetic_energy))
)

speed_monotonic = np.all(
    speed_differences <= 1e-12
)

energy_monotonic = np.all(
    energy_differences <= 1e-12
)

drag_dissipative = np.all(
    drag_power <= 1e-12
)

no_velocity_reversal = np.all(
    velocities[:, 0] >= -1e-12
)

no_cross_axis_motion = np.allclose(
    velocities[:, 1:],
    0.0,
    atol=1e-12,
    rtol=0.0,
)


# ---------------------------------------------------------------------
# Assertions
# ---------------------------------------------------------------------

assert all_finite
assert speed_monotonic
assert energy_monotonic
assert drag_dissipative
assert no_velocity_reversal
assert no_cross_axis_motion

assert speeds[-1] < speeds[0]
assert kinetic_energy[-1] < kinetic_energy[0]
assert positions[-1, 0] > positions[0, 0]


# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------

print("Passive hydrodynamic-drag validation")
print("-" * 55)

print(
    f"Initial speed:              "
    f"{speeds[0]:.12f} m/s"
)

print(
    f"Final speed:                "
    f"{speeds[-1]:.12f} m/s"
)

print(
    f"Speed reduction:            "
    f"{speeds[0] - speeds[-1]:.12f} m/s"
)

print()

print(
    f"Initial kinetic energy:     "
    f"{kinetic_energy[0]:.12f} J"
)

print(
    f"Final kinetic energy:       "
    f"{kinetic_energy[-1]:.12f} J"
)

print()

print(
    f"Initial x-position:         "
    f"{positions[0, 0]:.12f} m"
)

print(
    f"Final x-position:           "
    f"{positions[-1, 0]:.12f} m"
)

print()

print(
    f"Maximum speed increase:     "
    f"{np.max(speed_differences):.12e} m/s"
)

print(
    f"Maximum energy increase:    "
    f"{np.max(energy_differences):.12e} J"
)

print(
    f"Maximum drag power:         "
    f"{np.max(drag_power):.12e}"
)

print()

print(f"All states finite:          {all_finite}")
print(f"Speed monotonic:            {speed_monotonic}")
print(f"Energy monotonic:           {energy_monotonic}")
print(f"Drag always dissipative:    {drag_dissipative}")
print(f"No velocity reversal:       {no_velocity_reversal}")
print(f"No cross-axis motion:       {no_cross_axis_motion}")

print()
print("Passive drag integration checks passed.")

Passive hydrodynamic-drag validation
-------------------------------------------------------
Initial speed:              2.000000000000 m/s
Final speed:                0.587599691718 m/s
Speed reduction:            1.412400308282 m/s

Initial kinetic energy:     40.000000000000 J
Final kinetic energy:       3.452733977074 J

Initial x-position:         0.000000000000 m
Final x-position:           8.158639103114 m

Maximum speed increase:     -5.188250847098e-04 m/s
Maximum energy increase:    -6.099920991309e-03 J
Maximum drag power:         -6.086476261541e-01

All states finite:          True
Speed monotonic:            True
Energy monotonic:           True
Drag always dissipative:    True
No velocity reversal:       True
No cross-axis motion:       True

Passive drag integration checks passed.


## Step 4 - Validate Terminal Velocity Under Constant Applied Force

The passive-deceleration experiment confirmed that quadratic drag behaves correctly when connected to the inherited HydroLab-3D dynamics chain.

The next scientific validation target is the most important one in Notebook 03:

> **Does a vehicle under constant applied force approach the analytically predicted terminal velocity?**

For a one-axis constant applied force:

$$
F_A > 0
$$

and quadratic drag:

$$
F_D
=
-C_D |v|v
$$

the translational equation of motion is:

$$
m\dot{v}
=
F_A
-
C_D |v|v
$$

For positive motion:

$$
v > 0
$$

so:

$$
|v|v
=
v^2
$$

and therefore:

$$
m\dot{v}
=
F_A
-
C_Dv^2
$$

At terminal velocity:

$$
\dot{v}
=
0
$$

so the net force must vanish:

$$
F_A
-
C_Dv_\infty^2
=
0
$$

which gives:

$$
v_\infty
=
\sqrt{\frac{F_A}{C_D}}
$$

This provides an analytical equilibrium prediction that can be compared directly with the numerical HydroLab-3D simulation.

### Physical Evolution

Starting from rest:

$$
v_0=0
$$

the initial drag is zero:

$$
F_D(0)=0
$$

so initially:

$$
F_{net}=F_A
$$

and the vehicle accelerates.

As velocity increases:

$$
|F_D|
=
C_Dv^2
$$

also increases.

Therefore:

$$
F_{net}
=
F_A+F_D
$$

should progressively decrease toward zero.

Correspondingly:

$$
a
=
\frac{F_{net}}{m}
$$

should also decrease toward zero.

The expected progression is:

```text
constant applied force
        ↓
initial acceleration
        ↓
velocity increases
        ↓
drag magnitude increases
        ↓
net force decreases
        ↓
acceleration decreases
        ↓
velocity approaches terminal value
        ↓
applied force ≈ -drag force
```

### Validation Strategy

This experiment will use a deliberately simple one-axis case so that the analytical terminal-speed relation is unambiguous.

We will measure:

- analytical terminal speed;
- final numerical speed;
- absolute terminal-speed error;
- relative terminal-speed error;
- final applied force;
- final drag force;
- final net force;
- final acceleration;
- monotonicity of velocity;
- whether numerical velocity overshoots the analytical terminal speed;
- finite-value behavior.

The numerical velocity is not expected to equal the analytical terminal velocity exactly at finite simulation time.

Instead, the simulation should approach it asymptotically.

Therefore this step will report the actual numerical error rather than assume an arbitrary success tolerance in advance.

If the final state is clearly approaching the theoretical equilibrium without overshoot or instability, the measured convergence behavior will determine whether an additional timestep or duration study is necessary.

In [4]:
import numpy as np


# ---------------------------------------------------------------------
# Constant-force + quadratic-drag parameters
# ---------------------------------------------------------------------

mass = 20.0                   # kg
drag_coefficient = 3.0        # effective reduced-order coefficient

applied_force = np.array([
    12.0,
    0.0,
    0.0,
])

dt = 0.01                     # s
simulation_time = 30.0        # s

initial_position = np.array([
    0.0,
    0.0,
    -10.0,
])

initial_velocity = np.zeros(3)

num_steps = int(
    round(simulation_time / dt)
)


# ---------------------------------------------------------------------
# Analytical terminal speed
# ---------------------------------------------------------------------

applied_force_x = applied_force[0]

terminal_speed_analytical = np.sqrt(
    applied_force_x
    / drag_coefficient
)

print(
    "Analytical terminal speed:"
    f" {terminal_speed_analytical:.12f} m/s"
)


# ---------------------------------------------------------------------
# Allocate histories
# ---------------------------------------------------------------------

time = np.linspace(
    0.0,
    simulation_time,
    num_steps + 1,
)

positions = np.zeros(
    (num_steps + 1, 3)
)

velocities = np.zeros(
    (num_steps + 1, 3)
)

drag_forces = np.zeros(
    (num_steps + 1, 3)
)

net_forces = np.zeros(
    (num_steps + 1, 3)
)

accelerations = np.zeros(
    (num_steps + 1, 3)
)

speeds = np.zeros(
    num_steps + 1
)


positions[0] = initial_position
velocities[0] = initial_velocity
speeds[0] = np.linalg.norm(
    initial_velocity
)


# ---------------------------------------------------------------------
# Explicit-Euler simulation
# ---------------------------------------------------------------------

for k in range(num_steps):

    position_k = positions[k]
    velocity_k = velocities[k]

    drag_k = quadratic_drag_force(
        velocity_k,
        drag_coefficient,
    )

    net_force_k = (
        applied_force
        + drag_k
    )

    acceleration_k = (
        acceleration_from_force(
            net_force_k,
            mass,
        )
    )

    position_next = integrate_position(
        position_k,
        velocity_k,
        dt,
    )

    velocity_next = integrate_velocity(
        velocity_k,
        acceleration_k,
        dt,
    )

    drag_forces[k] = drag_k
    net_forces[k] = net_force_k
    accelerations[k] = acceleration_k

    positions[k + 1] = position_next
    velocities[k + 1] = velocity_next

    speeds[k + 1] = np.linalg.norm(
        velocity_next
    )


# ---------------------------------------------------------------------
# Evaluate forces at final state
# ---------------------------------------------------------------------

drag_forces[-1] = quadratic_drag_force(
    velocities[-1],
    drag_coefficient,
)

net_forces[-1] = (
    applied_force
    + drag_forces[-1]
)

accelerations[-1] = (
    acceleration_from_force(
        net_forces[-1],
        mass,
    )
)


# ---------------------------------------------------------------------
# Terminal-velocity error metrics
# ---------------------------------------------------------------------

final_speed = speeds[-1]

terminal_speed_absolute_error = abs(
    terminal_speed_analytical
    - final_speed
)

terminal_speed_relative_error = (
    terminal_speed_absolute_error
    / terminal_speed_analytical
)


# ---------------------------------------------------------------------
# Numerical-behavior checks
# ---------------------------------------------------------------------

speed_differences = np.diff(
    speeds
)

all_finite = (
    np.all(np.isfinite(positions))
    and np.all(np.isfinite(velocities))
    and np.all(np.isfinite(drag_forces))
    and np.all(np.isfinite(net_forces))
    and np.all(np.isfinite(accelerations))
    and np.all(np.isfinite(speeds))
)

speed_monotonic = np.all(
    speed_differences >= -1e-12
)

terminal_speed_not_overshot = (
    np.max(speeds)
    <= terminal_speed_analytical + 1e-12
)

no_cross_axis_motion = (
    np.allclose(
        velocities[:, 1:],
        0.0,
        atol=1e-12,
        rtol=0.0,
    )
)

drag_opposes_motion = np.all(
    np.einsum(
        "ij,ij->i",
        drag_forces,
        velocities,
    )
    <= 1e-12
)


# ---------------------------------------------------------------------
# Assertions for fundamental physical behavior
# ---------------------------------------------------------------------

assert all_finite
assert speed_monotonic
assert terminal_speed_not_overshot
assert no_cross_axis_motion
assert drag_opposes_motion

assert final_speed > 0.0

assert abs(
    net_forces[-1, 0]
) < abs(
    net_forces[0, 0]
)

assert abs(
    accelerations[-1, 0]
) < abs(
    accelerations[0, 0]
)


# ---------------------------------------------------------------------
# Force-balance diagnostics
# ---------------------------------------------------------------------

initial_net_force_x = (
    net_forces[0, 0]
)

final_drag_force_x = (
    drag_forces[-1, 0]
)

final_net_force_x = (
    net_forces[-1, 0]
)

final_acceleration_x = (
    accelerations[-1, 0]
)

force_balance_error = abs(
    applied_force_x
    + final_drag_force_x
)


# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------

print()
print(
    "Constant-force terminal-velocity validation"
)

print("-" * 60)

print(
    f"Mass:                        "
    f"{mass:.6f} kg"
)

print(
    f"Applied force:               "
    f"{applied_force_x:.6f} N"
)

print(
    f"Drag coefficient:            "
    f"{drag_coefficient:.6f}"
)

print()

print(
    f"Analytical terminal speed:   "
    f"{terminal_speed_analytical:.12f} m/s"
)

print(
    f"Final numerical speed:       "
    f"{final_speed:.12f} m/s"
)

print(
    f"Absolute speed error:        "
    f"{terminal_speed_absolute_error:.12e} m/s"
)

print(
    f"Relative speed error:        "
    f"{terminal_speed_relative_error:.12e}"
)

print()

print(
    f"Initial net force:           "
    f"{initial_net_force_x:.12f} N"
)

print(
    f"Final drag force:            "
    f"{final_drag_force_x:.12f} N"
)

print(
    f"Final net force:             "
    f"{final_net_force_x:.12e} N"
)

print(
    f"Force-balance error:         "
    f"{force_balance_error:.12e} N"
)

print(
    f"Final acceleration:          "
    f"{final_acceleration_x:.12e} m/s^2"
)

print()

print(
    f"Maximum simulated speed:     "
    f"{np.max(speeds):.12f} m/s"
)

print(
    f"Final x-position:            "
    f"{positions[-1, 0]:.12f} m"
)

print()

print(
    f"All states finite:           "
    f"{all_finite}"
)

print(
    f"Speed monotonic:             "
    f"{speed_monotonic}"
)

print(
    f"No terminal-speed overshoot: "
    f"{terminal_speed_not_overshot}"
)

print(
    f"No cross-axis motion:        "
    f"{no_cross_axis_motion}"
)

print(
    f"Drag opposes motion:         "
    f"{drag_opposes_motion}"
)

print()
print(
    "Terminal-velocity simulation completed."
)

Analytical terminal speed: 2.000000000000 m/s

Constant-force terminal-velocity validation
------------------------------------------------------------
Mass:                        20.000000 kg
Applied force:               12.000000 N
Drag coefficient:            3.000000

Analytical terminal speed:   2.000000000000 m/s
Final numerical speed:       1.999999942053 m/s
Absolute speed error:        5.794670432380e-08 m/s
Relative speed error:        2.897335216190e-08

Initial net force:           12.000000000000 N
Final drag force:            -11.999999304640 N
Final net force:             6.953604412274e-07 N
Force-balance error:         6.953604412274e-07 N
Final acceleration:          3.476802206137e-08 m/s^2

Maximum simulated speed:     1.999999942053 m/s
Final x-position:            55.375149810591 m

All states finite:           True
Speed monotonic:             True
No terminal-speed overshoot: True
No cross-axis motion:        True
Drag opposes motion:         True

Terminal-vel

## Step 5 - Validate the Full Velocity Transient Against the Analytical Solution

The constant-force experiment confirmed that the numerical model approaches the correct terminal speed and force balance.

However, terminal convergence alone does not completely validate the time evolution.

A simulation run for sufficiently long time may arrive very close to the correct equilibrium even while accumulating noticeable transient integration error.

Because the present one-dimensional quadratic-drag problem has a closed-form analytical solution, we can compare the **entire numerical velocity trajectory** against that solution.

For positive velocity under constant positive applied force:

$$
m\dot{v}
=
F_A
-
C_Dv^2
$$

Define the terminal speed:

$$
v_\infty
=
\sqrt{\frac{F_A}{C_D}}
$$

For the initial condition:

$$
v(0)=0
$$

the analytical solution is:

$$
v(t)
=
v_\infty
\tanh
\left(
\frac{\sqrt{F_AC_D}}{m}t
\right)
$$

Equivalently, defining:

$$
\lambda
=
\frac{\sqrt{F_AC_D}}{m}
$$

gives:

$$
v(t)
=
v_\infty\tanh(\lambda t)
$$

This solution contains the complete acceleration-to-terminal-speed transient.

### Why This Validation Is Stronger

The terminal-speed test answered:

> Does the numerical system approach the correct equilibrium?

This step answers:

> Does the numerical integrator reproduce the correct physical evolution on the way to that equilibrium?

We will compare the explicit-Euler velocity history already generated in Step 4 against the analytical velocity evaluated at exactly the same timestamps.

For each sample:

$$
e_v(t_k)
=
v_{numerical}(t_k)
-
v_{analytical}(t_k)
$$

We will measure:

- maximum absolute velocity error over the entire trajectory;
- final velocity error;
- time at which the maximum absolute error occurs;
- root-mean-square velocity error;
- finite-value behavior;
- agreement at the initial condition;
- agreement with the same analytical terminal speed already validated.

No new physical model is introduced in this step.

The purpose is to characterize the numerical accuracy of the existing explicit-Euler simulation before deciding whether a dedicated timestep-refinement experiment is scientifically necessary.

In [5]:
# ---------------------------------------------------------------------
# Analytical transient for constant force + quadratic drag
#
# For:
#
#     m dv/dt = F_A - C_D v^2
#
# with v(0) = 0 and positive motion:
#
#     v(t) = v_inf * tanh(lambda * t)
#
# where:
#
#     v_inf = sqrt(F_A / C_D)
#     lambda = sqrt(F_A * C_D) / m
# ---------------------------------------------------------------------

lambda_rate = (
    np.sqrt(
        applied_force_x
        * drag_coefficient
    )
    / mass
)

analytical_velocity_x = (
    terminal_speed_analytical
    * np.tanh(
        lambda_rate * time
    )
)


# ---------------------------------------------------------------------
# Numerical velocity from Step 4
# ---------------------------------------------------------------------

numerical_velocity_x = (
    velocities[:, 0]
)


# ---------------------------------------------------------------------
# Error history
# ---------------------------------------------------------------------

velocity_error = (
    numerical_velocity_x
    - analytical_velocity_x
)

absolute_velocity_error = np.abs(
    velocity_error
)

maximum_error_index = np.argmax(
    absolute_velocity_error
)

maximum_absolute_velocity_error = (
    absolute_velocity_error[
        maximum_error_index
    ]
)

maximum_error_time = (
    time[maximum_error_index]
)

final_velocity_error = (
    velocity_error[-1]
)

rmse_velocity = np.sqrt(
    np.mean(
        velocity_error ** 2
    )
)


# ---------------------------------------------------------------------
# Additional analytical checks
# ---------------------------------------------------------------------

analytical_initial_condition_correct = (
    np.isclose(
        analytical_velocity_x[0],
        0.0,
        atol=1e-15,
        rtol=0.0,
    )
)

analytical_values_finite = np.all(
    np.isfinite(
        analytical_velocity_x
    )
)

error_values_finite = np.all(
    np.isfinite(
        velocity_error
    )
)

analytical_speed_monotonic = np.all(
    np.diff(
        analytical_velocity_x
    )
    >= -1e-12
)

analytical_terminal_bound = np.all(
    analytical_velocity_x
    <= terminal_speed_analytical + 1e-12
)


# ---------------------------------------------------------------------
# Assertions
# ---------------------------------------------------------------------

assert analytical_initial_condition_correct
assert analytical_values_finite
assert error_values_finite
assert analytical_speed_monotonic
assert analytical_terminal_bound

assert np.isclose(
    analytical_velocity_x[0],
    numerical_velocity_x[0],
    atol=1e-15,
    rtol=0.0,
)


# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------

print(
    "Analytical transient velocity validation"
)

print("-" * 60)

print(
    f"Terminal speed:              "
    f"{terminal_speed_analytical:.12f} m/s"
)

print(
    f"Transient rate lambda:       "
    f"{lambda_rate:.12f} 1/s"
)

print()

print(
    f"Maximum absolute error:      "
    f"{maximum_absolute_velocity_error:.12e} m/s"
)

print(
    f"Maximum-error time:          "
    f"{maximum_error_time:.6f} s"
)

print(
    f"Final velocity error:        "
    f"{final_velocity_error:.12e} m/s"
)

print(
    f"Velocity RMSE:               "
    f"{rmse_velocity:.12e} m/s"
)

print()

print(
    f"Numerical velocity at "
    f"max-error time:      "
    f"{numerical_velocity_x[maximum_error_index]:.12f} m/s"
)

print(
    f"Analytical velocity at "
    f"max-error time:     "
    f"{analytical_velocity_x[maximum_error_index]:.12f} m/s"
)

print()

print(
    f"Analytical initial condition correct: "
    f"{analytical_initial_condition_correct}"
)

print(
    f"Analytical values finite:             "
    f"{analytical_values_finite}"
)

print(
    f"Error values finite:                  "
    f"{error_values_finite}"
)

print(
    f"Analytical speed monotonic:           "
    f"{analytical_speed_monotonic}"
)

print(
    f"Analytical terminal bound respected:  "
    f"{analytical_terminal_bound}"
)

print()
print(
    "Full transient comparison completed."
)

Analytical transient velocity validation
------------------------------------------------------------
Terminal speed:              2.000000000000 m/s
Transient rate lambda:       0.300000000000 1/s

Maximum absolute error:      1.104717980683e-03 m/s
Maximum-error time:          3.620000 s
Final velocity error:        2.973213897306e-09 m/s
Velocity RMSE:               3.943739891567e-04 m/s

Numerical velocity at max-error time:      1.591931803355 m/s
Analytical velocity at max-error time:     1.590827085374 m/s

Analytical initial condition correct: True
Analytical values finite:             True
Error values finite:                  True
Analytical speed monotonic:           True
Analytical terminal bound respected:  True

Full transient comparison completed.


## Step 6 - Verify Timestep Convergence of the Drag Transient

The analytical transient comparison showed that the explicit-Euler simulation reproduces the physical velocity evolution closely, but not exactly.

For the current timestep:

$$
\Delta t
=
0.01\ \text{s}
$$

the maximum transient velocity error was approximately:

$$
1.10\times10^{-3}\ \text{m/s}
$$

This discrepancy is expected because quadratic drag produces state-dependent acceleration:

$$
a(v)
=
\frac{F_A-C_Dv^2}{m}
$$

and explicit Euler approximates this continuously changing acceleration using its beginning-of-step value.

The purpose of this step is not to begin a broad numerical-method study.

Instead, we will answer one focused question:

> **Does the transient velocity error decrease systematically when the timestep is refined?**

For a first-order explicit-Euler method, we expect the global discretization error to scale approximately as:

$$
E(\Delta t)
=
O(\Delta t)
$$

Therefore, when the timestep is halved:

$$
\Delta t
\rightarrow
\frac{\Delta t}{2}
$$

the error should approximately halve once the simulation is operating in the expected convergence regime.

Equivalently, for successive timestep pairs:

$$
\frac{E(\Delta t)}
{E(\Delta t/2)}
\approx
2
$$

### Experiment

We will repeat the same constant-force quadratic-drag transient using:

- $\Delta t = 0.04$ s;
- $\Delta t = 0.02$ s;
- $\Delta t = 0.01$ s;
- $\Delta t = 0.005$ s.

All physical parameters and the simulation duration will remain unchanged.

For each timestep, the numerical velocity will be compared directly against:

$$
v(t)
=
v_\infty
\tanh(\lambda t)
$$

at the corresponding numerical timestamps.

The primary error metric will be:

$$
E_{\max}
=
\max_k
\left|
v_{numerical}(t_k)
-
v_{analytical}(t_k)
\right|
$$

We will then compute successive error ratios.

### Validation Objective

This step will be considered successful if:

- all simulations remain finite;
- no terminal-speed overshoot occurs;
- maximum transient error decreases as the timestep is refined;
- successive error ratios are consistent with approximately first-order convergence.

No exact ratio of `2.0` will be required.

The measured ratios themselves will determine whether the expected explicit-Euler convergence behavior is present.

If this check succeeds, further numerical-method analysis will not be necessary for Notebook 03.

In [6]:
# ---------------------------------------------------------------------
# Timestep-refinement study for the constant-force drag transient
# ---------------------------------------------------------------------

timesteps = np.array([
    0.04,
    0.02,
    0.01,
    0.005,
])

simulation_time_refinement = 15.0

maximum_errors = []
rmse_errors = []
final_errors = []
finite_flags = []
overshoot_flags = []


for dt_test in timesteps:

    num_steps_test = int(
        round(
            simulation_time_refinement
            / dt_test
        )
    )

    time_test = np.linspace(
        0.0,
        simulation_time_refinement,
        num_steps_test + 1,
    )

    velocity_test = np.zeros(
        (num_steps_test + 1, 3)
    )

    # -------------------------------------------------------------
    # Numerical explicit-Euler integration
    # -------------------------------------------------------------

    for k in range(num_steps_test):

        drag_k = quadratic_drag_force(
            velocity_test[k],
            drag_coefficient,
        )

        net_force_k = (
            applied_force
            + drag_k
        )

        acceleration_k = (
            acceleration_from_force(
                net_force_k,
                mass,
            )
        )

        velocity_test[k + 1] = (
            integrate_velocity(
                velocity_test[k],
                acceleration_k,
                dt_test,
            )
        )

    numerical_vx_test = (
        velocity_test[:, 0]
    )

    # -------------------------------------------------------------
    # Analytical transient at the same timestamps
    # -------------------------------------------------------------

    analytical_vx_test = (
        terminal_speed_analytical
        * np.tanh(
            lambda_rate
            * time_test
        )
    )

    error_test = (
        numerical_vx_test
        - analytical_vx_test
    )

    absolute_error_test = np.abs(
        error_test
    )

    # -------------------------------------------------------------
    # Metrics
    # -------------------------------------------------------------

    maximum_errors.append(
        np.max(
            absolute_error_test
        )
    )

    rmse_errors.append(
        np.sqrt(
            np.mean(
                error_test ** 2
            )
        )
    )

    final_errors.append(
        abs(
            error_test[-1]
        )
    )

    finite_flags.append(
        np.all(
            np.isfinite(
                velocity_test
            )
        )
    )

    overshoot_flags.append(
        np.max(
            numerical_vx_test
        )
        <= terminal_speed_analytical
        + 1e-12
    )


maximum_errors = np.array(
    maximum_errors
)

rmse_errors = np.array(
    rmse_errors
)

final_errors = np.array(
    final_errors
)


# ---------------------------------------------------------------------
# Successive convergence ratios
#
# E(dt) / E(dt/2)
# ---------------------------------------------------------------------

error_ratios = (
    maximum_errors[:-1]
    / maximum_errors[1:]
)


# ---------------------------------------------------------------------
# Observed convergence order
#
# p = log(E_h / E_h/2) / log(2)
# ---------------------------------------------------------------------

observed_orders = (
    np.log(
        error_ratios
    )
    / np.log(2.0)
)


# ---------------------------------------------------------------------
# Validation
# ---------------------------------------------------------------------

errors_decrease = np.all(
    np.diff(
        maximum_errors
    )
    < 0.0
)

all_finite_refinement = all(
    finite_flags
)

no_overshoot_refinement = all(
    overshoot_flags
)


assert all_finite_refinement
assert no_overshoot_refinement
assert errors_decrease


# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------

print(
    "Quadratic-drag timestep refinement"
)

print("-" * 78)

print(
    f"{'dt [s]':>10}"
    f"{'max error [m/s]':>22}"
    f"{'RMSE [m/s]':>20}"
    f"{'final error [m/s]':>22}"
)

print("-" * 78)


for dt_test, max_error, rmse, final_error in zip(
    timesteps,
    maximum_errors,
    rmse_errors,
    final_errors,
):

    print(
        f"{dt_test:10.5f}"
        f"{max_error:22.12e}"
        f"{rmse:20.12e}"
        f"{final_error:22.12e}"
    )


print()
print(
    "Successive maximum-error ratios:"
)

for i, ratio in enumerate(
    error_ratios
):

    print(
        f"dt {timesteps[i]:.5f} -> "
        f"{timesteps[i + 1]:.5f}: "
        f"{ratio:.12f}"
    )


print()
print(
    "Observed convergence orders:"
)

for i, order in enumerate(
    observed_orders
):

    print(
        f"dt {timesteps[i]:.5f} -> "
        f"{timesteps[i + 1]:.5f}: "
        f"{order:.12f}"
    )


print()
print(
    f"Errors decrease with refinement: "
    f"{errors_decrease}"
)

print(
    f"All simulations finite:          "
    f"{all_finite_refinement}"
)

print(
    f"No terminal-speed overshoot:     "
    f"{no_overshoot_refinement}"
)

print()
print(
    "Timestep-refinement study completed."
)

Quadratic-drag timestep refinement
------------------------------------------------------------------------------
    dt [s]       max error [m/s]          RMSE [m/s]     final error [m/s]
------------------------------------------------------------------------------
   0.04000    4.431825863088e-03  2.231018602674e-03    4.372742573566e-05
   0.02000    2.211606988392e-03  1.115340374985e-03    2.220316422297e-05
   0.01000    1.104717980683e-03  5.576290895092e-04    1.118702570335e-05
   0.00500    5.520889562456e-04  2.788044110247e-04    5.614945943089e-06

Successive maximum-error ratios:
dt 0.04000 -> 0.02000: 2.003893949671
dt 0.02000 -> 0.01000: 2.001965231909
dt 0.01000 -> 0.00500: 2.000978226761

Observed convergence orders:
dt 0.04000 -> 0.02000: 1.002806160064
dt 0.02000 -> 0.01000: 1.001416919135
dt 0.01000 -> 0.00500: 1.000705468936

Errors decrease with refinement: True
All simulations finite:          True
No terminal-speed overshoot:     True

Timestep-refinement stud

## Step 7 - Promote the Validated Drag Model into Production Code

The quadratic drag model has now passed the required notebook-level scientific validation.

The validated evidence includes:

- zero velocity produces zero drag;
- zero coefficient produces zero drag;
- drag acts opposite motion;
- velocity reversal reverses drag;
- drag magnitude scales quadratically with speed;
- arbitrary 3D velocity produces anti-parallel drag;
- invalid inputs are rejected;
- passive motion decelerates monotonically;
- kinetic energy decreases monotonically under passive drag;
- constant applied force approaches the analytical terminal speed;
- the full numerical velocity transient agrees closely with the analytical solution;
- explicit-Euler transient error decreases with approximately first-order convergence.

The timestep-refinement study produced successive maximum-error ratios of approximately:

$$
2.0039,\quad
2.0020,\quad
2.0010
$$

with observed convergence orders approximately:

$$
1.0028,\quad
1.0014,\quad
1.0007
$$

This is consistent with the expected first-order behavior of the inherited explicit-Euler integration scheme.

The notebook-local drag implementation is therefore sufficiently validated to become reusable HydroLab-3D production code.

### Production Target

The repository audit established that:

`src/hydrolab3d/dynamics/drag.py`

was an empty placeholder.

The validated implementation will now be promoted there as:

`quadratic_drag_force()`

The public dynamics package interface:

`src/hydrolab3d/dynamics/__init__.py`

will also expose the new drag function alongside the previously validated:

- `integrate_position()`;
- `acceleration_from_force()`;
- `integrate_velocity()`.

### Regression-Test Philosophy

The scientific notebook and the automated test suite serve different purposes.

The notebook provides the detailed physical and numerical validation record.

The production test suite should provide compact regression protection against accidental future changes.

Focused drag tests will cover:

- zero velocity;
- zero coefficient;
- a known one-axis result;
- drag opposition to velocity;
- odd symmetry;
- quadratic speed scaling;
- arbitrary 3D direction behavior;
- finite outputs;
- invalid velocity shape;
- nonfinite velocity;
- nonscalar coefficient;
- nonfinite coefficient;
- negative coefficient.

The new drag tests will then be executed together with the inherited dynamics tests.

### Test-Environment Note

HydroLab-3D currently uses a `src/` package layout:

```text
HydroLab-3D/
├── src/
│   └── hydrolab3d/
└── tests/
```

The notebook can import the package because `src/` has already been added to the notebook process's Python search path.

However, `pytest` is launched as a separate Python process.

That subprocess does not automatically inherit the notebook's modified `sys.path`.

Therefore the regression-test subprocess will explicitly receive:

`PYTHONPATH=<repository>/src`

so that it imports the same production package being developed in this repository.

This changes only the test execution environment.

It does not modify the physical model or production API.

This step is successful only if both the new drag regression tests and the inherited dynamics regression tests pass.

In [8]:
from pathlib import Path
import os
import subprocess
import sys


# ---------------------------------------------------------------------
# Resolve repository paths
# ---------------------------------------------------------------------

repo_root = Path.cwd().resolve()

if repo_root.name == "notebooks":
    repo_root = repo_root.parent


src_path = (
    repo_root
    / "src"
)

drag_module_path = (
    src_path
    / "hydrolab3d"
    / "dynamics"
    / "drag.py"
)

dynamics_init_path = (
    src_path
    / "hydrolab3d"
    / "dynamics"
    / "__init__.py"
)

drag_test_path = (
    repo_root
    / "tests"
    / "test_drag.py"
)


# ---------------------------------------------------------------------
# Production drag module
# ---------------------------------------------------------------------

drag_module_contents = '''"""Hydrodynamic drag models for HydroLab-3D."""

import numpy as np


def quadratic_drag_force(velocity, drag_coefficient):
    """
    Compute reduced-order quadratic translational drag.

    The model is:

        F_D = -C_D * ||v|| * v

    where ``drag_coefficient`` is an effective reduced-order
    coefficient rather than a dimensionless textbook drag coefficient.

    Parameters
    ----------
    velocity : array-like
        3D translational velocity [vx, vy, vz] in m/s.
    drag_coefficient : float
        Nonnegative effective quadratic drag coefficient.

    Returns
    -------
    np.ndarray
        Drag-force vector [Fx, Fy, Fz] in newtons.
    """
    velocity = np.asarray(
        velocity,
        dtype=float,
    )

    if velocity.shape != (3,):
        raise ValueError(
            "velocity must have shape (3,)"
        )

    if not np.all(
        np.isfinite(velocity)
    ):
        raise ValueError(
            "velocity must contain only finite values"
        )

    if not np.isscalar(
        drag_coefficient
    ):
        raise ValueError(
            "drag_coefficient must be a scalar"
        )

    drag_coefficient = float(
        drag_coefficient
    )

    if not np.isfinite(
        drag_coefficient
    ):
        raise ValueError(
            "drag_coefficient must be finite"
        )

    if drag_coefficient < 0.0:
        raise ValueError(
            "drag_coefficient must be nonnegative"
        )

    speed = np.linalg.norm(
        velocity
    )

    return (
        -drag_coefficient
        * speed
        * velocity
    )
'''


# ---------------------------------------------------------------------
# Dynamics package exports
# ---------------------------------------------------------------------

dynamics_init_contents = '''"""Dynamics utilities for HydroLab-3D."""

from .drag import quadratic_drag_force
from .kinematics import integrate_position
from .rigid_body import acceleration_from_force, integrate_velocity

__all__ = [
    "quadratic_drag_force",
    "integrate_position",
    "acceleration_from_force",
    "integrate_velocity",
]
'''


# ---------------------------------------------------------------------
# Focused production drag regression tests
# ---------------------------------------------------------------------

drag_test_contents = '''import numpy as np
import pytest

from hydrolab3d.dynamics.drag import quadratic_drag_force


def test_zero_velocity_produces_zero_drag():
    drag = quadratic_drag_force(
        np.zeros(3),
        2.5,
    )

    assert np.array_equal(
        drag,
        np.zeros(3),
    )


def test_zero_coefficient_produces_zero_drag():
    drag = quadratic_drag_force(
        np.array([1.0, -2.0, 0.5]),
        0.0,
    )

    assert np.array_equal(
        drag,
        np.zeros(3),
    )


def test_known_one_axis_drag():
    drag = quadratic_drag_force(
        np.array([3.0, 0.0, 0.0]),
        2.0,
    )

    expected = np.array([
        -18.0,
        0.0,
        0.0,
    ])

    assert np.allclose(
        drag,
        expected,
        atol=1e-12,
        rtol=0.0,
    )


def test_drag_opposes_velocity():
    velocity = np.array([
        2.0,
        -1.0,
        0.5,
    ])

    drag = quadratic_drag_force(
        velocity,
        1.7,
    )

    assert np.dot(
        drag,
        velocity,
    ) < 0.0


def test_drag_has_odd_symmetry():
    velocity = np.array([
        1.25,
        -2.5,
        0.75,
    ])

    drag_forward = quadratic_drag_force(
        velocity,
        0.8,
    )

    drag_reverse = quadratic_drag_force(
        -velocity,
        0.8,
    )

    assert np.allclose(
        drag_reverse,
        -drag_forward,
        atol=1e-12,
        rtol=0.0,
    )


def test_drag_scales_quadratically_with_speed():
    velocity = np.array([
        2.0,
        -1.0,
        0.5,
    ])

    drag_base = quadratic_drag_force(
        velocity,
        1.7,
    )

    drag_double = quadratic_drag_force(
        2.0 * velocity,
        1.7,
    )

    ratio = (
        np.linalg.norm(drag_double)
        / np.linalg.norm(drag_base)
    )

    assert np.isclose(
        ratio,
        4.0,
        atol=1e-12,
        rtol=0.0,
    )


def test_drag_is_antiparallel_to_3d_velocity():
    velocity = np.array([
        1.25,
        -2.5,
        0.75,
    ])

    drag = quadratic_drag_force(
        velocity,
        0.8,
    )

    velocity_direction = (
        velocity
        / np.linalg.norm(velocity)
    )

    drag_direction = (
        drag
        / np.linalg.norm(drag)
    )

    assert np.allclose(
        drag_direction,
        -velocity_direction,
        atol=1e-12,
        rtol=0.0,
    )


def test_drag_output_is_finite():
    drag = quadratic_drag_force(
        np.array([1.0, -2.0, 3.0]),
        1.5,
    )

    assert np.all(
        np.isfinite(drag)
    )


def test_invalid_velocity_shape_rejected():
    with pytest.raises(ValueError):
        quadratic_drag_force(
            np.ones(2),
            1.0,
        )


@pytest.mark.parametrize(
    "velocity",
    [
        np.array([1.0, np.nan, 0.0]),
        np.array([1.0, np.inf, 0.0]),
    ],
)
def test_nonfinite_velocity_rejected(
    velocity,
):
    with pytest.raises(ValueError):
        quadratic_drag_force(
            velocity,
            1.0,
        )


def test_nonscalar_coefficient_rejected():
    with pytest.raises(ValueError):
        quadratic_drag_force(
            np.ones(3),
            [1.0],
        )


@pytest.mark.parametrize(
    "drag_coefficient",
    [
        np.nan,
        np.inf,
        -1.0,
    ],
)
def test_invalid_coefficient_rejected(
    drag_coefficient,
):
    with pytest.raises(ValueError):
        quadratic_drag_force(
            np.ones(3),
            drag_coefficient,
        )
'''


# ---------------------------------------------------------------------
# Write validated production implementation
#
# This is intentionally idempotent:
# rerunning the cell replaces the same three files with the same
# validated contents.
# ---------------------------------------------------------------------

drag_module_path.write_text(
    drag_module_contents,
    encoding="utf-8",
)

dynamics_init_path.write_text(
    dynamics_init_contents,
    encoding="utf-8",
)

drag_test_path.write_text(
    drag_test_contents,
    encoding="utf-8",
)


print("Production files updated:")

print(
    drag_module_path.relative_to(
        repo_root
    )
)

print(
    dynamics_init_path.relative_to(
        repo_root
    )
)

print(
    drag_test_path.relative_to(
        repo_root
    )
)

print()


# ---------------------------------------------------------------------
# Construct subprocess environment
#
# The notebook process knows about src/ through sys.path, but a new
# pytest subprocess does not inherit those runtime sys.path changes.
#
# Explicitly expose the repository's src/ directory through PYTHONPATH.
# ---------------------------------------------------------------------

pytest_environment = os.environ.copy()

existing_pythonpath = pytest_environment.get(
    "PYTHONPATH",
    "",
)

if existing_pythonpath:
    pytest_environment["PYTHONPATH"] = (
        str(src_path)
        + os.pathsep
        + existing_pythonpath
    )
else:
    pytest_environment["PYTHONPATH"] = str(
        src_path
    )


# ---------------------------------------------------------------------
# Run new drag tests together with inherited dynamics tests
# ---------------------------------------------------------------------

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests/test_drag.py",
        "tests/test_dynamics.py",
        "-q",
    ],
    cwd=repo_root,
    env=pytest_environment,
    capture_output=True,
    text=True,
)


print("pytest output")
print("-" * 60)

print(
    result.stdout
)


if result.stderr.strip():

    print("pytest stderr")
    print("-" * 60)

    print(
        result.stderr
    )


# ---------------------------------------------------------------------
# Validate pytest result
# ---------------------------------------------------------------------

assert result.returncode == 0, (
    "Production regression tests failed."
)


print(
    "Production drag promotion and "
    "regression validation passed."
)

Production files updated:
src/hydrolab3d/dynamics/drag.py
src/hydrolab3d/dynamics/__init__.py
tests/test_drag.py

pytest output
------------------------------------------------------------
...............................                                          [100%]
31 passed in 0.06s

Production drag promotion and regression validation passed.


## Step 8 - Re-import Production Dynamics, Re-run the Canonical Validation, and Generate Persistent Artifacts

The validated drag implementation has now been promoted into:

`src/hydrolab3d/dynamics/drag.py`

and the package interface:

`src/hydrolab3d/dynamics/__init__.py`

has been updated to expose:

`quadratic_drag_force()`

The combined production regression suite reports:

`31 passed`

so the promoted implementation itself is valid.

### Jupyter Import-Cache Consideration

Earlier notebook steps imported:

`hydrolab3d.dynamics`

before the package `__init__.py` was updated.

Python caches imported modules inside the running interpreter.

Therefore, changing the source file on disk does not automatically reconstruct an already imported package object inside the current Jupyter kernel.

Before performing the canonical production-code validation, this step will explicitly:

1. invalidate Python's filesystem import caches;
2. reload `hydrolab3d.dynamics`;
3. verify that `quadratic_drag_force` is exposed by the production package;
4. use that reloaded production API for the final canonical simulation.

This is a notebook-runtime issue only.

It does not represent a defect in the promoted drag implementation, as independently confirmed by the successful production regression run.

### Canonical Experiment

The final saved validation uses:

$$
m
=
20\ \text{kg}
$$

$$
F_A
=
12\ \text{N}
$$

$$
C_D
=
3
$$

$$
\Delta t
=
0.01\ \text{s}
$$

and:

$$
T
=
30\ \text{s}
$$

with the vehicle starting from rest.

The predicted terminal speed is:

$$
v_\infty
=
\sqrt{\frac{F_A}{C_D}}
=
2\ \text{m/s}
$$

The production simulation should reproduce the previously validated behavior:

- monotonic acceleration toward terminal speed;
- increasing opposing drag magnitude;
- decreasing net force;
- decreasing acceleration;
- no terminal-speed overshoot;
- no cross-axis motion;
- finite numerical state throughout the simulation.

### Analytical Transient Reference

For this positive one-axis case starting from rest:

$$
v(t)
=
v_\infty
\tanh
\left(
\frac{\sqrt{F_AC_D}}{m}t
\right)
$$

The production simulation will again be compared against this analytical transient so that the persistent artifacts record both:

- equilibrium correctness;
- transient numerical accuracy.

### Persistent Numerical Artifact

The canonical CSV will contain:

- time;
- applied-force components;
- drag-force components;
- net-force components;
- acceleration components;
- velocity components;
- scalar speed;
- position components;
- analytical transient velocity;
- analytical terminal speed;
- signed transient velocity error;
- absolute terminal-speed error.

### Persistent Figures

Two figures will be preserved.

#### Velocity Validation

This figure compares:

- production numerical velocity;
- analytical transient velocity;
- analytical terminal speed.

#### Force Balance

This figure shows:

- applied force;
- hydrodynamic drag force;
- net force.

Together these demonstrate both the numerical trajectory and the physical force-balance mechanism responsible for terminal velocity.

### Output Location

The canonical Notebook 03 artifacts will be:

- `results/notebooks/03_hydrodynamic_drag/data/terminal_velocity_validation.csv`
- `results/notebooks/03_hydrodynamic_drag/plots/velocity_validation.png`
- `results/notebooks/03_hydrodynamic_drag/plots/force_balance.png`

After writing them, the notebook will read the CSV back and verify that every expected artifact exists and is nonempty.

In [10]:
import importlib
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


# ---------------------------------------------------------------------
# Resolve HydroLab-3D repository
# ---------------------------------------------------------------------

repo_root = Path.cwd().resolve()

if repo_root.name == "notebooks":
    repo_root = repo_root.parent

src_path = repo_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))


# ---------------------------------------------------------------------
# Refresh the production dynamics package
#
# hydrolab3d.dynamics was imported earlier in this notebook before
# Step 7 modified its __init__.py. Reload it so this running kernel
# sees the newly exported quadratic_drag_force().
# ---------------------------------------------------------------------

importlib.invalidate_caches()

import hydrolab3d.dynamics as dynamics

dynamics = importlib.reload(
    dynamics
)


# ---------------------------------------------------------------------
# Verify required production API
# ---------------------------------------------------------------------

required_production_symbols = [
    "quadratic_drag_force",
    "acceleration_from_force",
    "integrate_position",
    "integrate_velocity",
]

missing_symbols = [
    name
    for name in required_production_symbols
    if not hasattr(dynamics, name)
]

assert not missing_symbols, (
    "Production dynamics API is missing: "
    + ", ".join(missing_symbols)
)


production_quadratic_drag_force = (
    dynamics.quadratic_drag_force
)

acceleration_from_force = (
    dynamics.acceleration_from_force
)

integrate_position = (
    dynamics.integrate_position
)

integrate_velocity = (
    dynamics.integrate_velocity
)


print("Production dynamics API refreshed.")

print(
    "Available required symbols:",
    required_production_symbols,
)

print()


# ---------------------------------------------------------------------
# Canonical Notebook 03 parameters
# ---------------------------------------------------------------------

mass = 20.0
drag_coefficient = 3.0

applied_force = np.array([
    12.0,
    0.0,
    0.0,
])

dt = 0.01
simulation_time = 30.0

initial_position = np.array([
    0.0,
    0.0,
    -10.0,
])

initial_velocity = np.zeros(3)

num_steps = int(
    round(
        simulation_time / dt
    )
)


# ---------------------------------------------------------------------
# Analytical references
# ---------------------------------------------------------------------

applied_force_x = (
    applied_force[0]
)

terminal_speed_analytical = np.sqrt(
    applied_force_x
    / drag_coefficient
)

lambda_rate = (
    np.sqrt(
        applied_force_x
        * drag_coefficient
    )
    / mass
)


# ---------------------------------------------------------------------
# Allocate canonical histories
# ---------------------------------------------------------------------

time = np.linspace(
    0.0,
    simulation_time,
    num_steps + 1,
)

positions = np.zeros(
    (num_steps + 1, 3)
)

velocities = np.zeros(
    (num_steps + 1, 3)
)

drag_forces = np.zeros(
    (num_steps + 1, 3)
)

net_forces = np.zeros(
    (num_steps + 1, 3)
)

accelerations = np.zeros(
    (num_steps + 1, 3)
)

speeds = np.zeros(
    num_steps + 1
)

applied_forces = np.repeat(
    applied_force[None, :],
    num_steps + 1,
    axis=0,
)


positions[0] = (
    initial_position
)

velocities[0] = (
    initial_velocity
)


# ---------------------------------------------------------------------
# Canonical simulation using production HydroLab-3D code
#
# Explicit-Euler ordering:
#
# 1. drag from v_k
# 2. net force
# 3. acceleration
# 4. position from v_k
# 5. velocity from a_k
# ---------------------------------------------------------------------

for k in range(num_steps):

    position_k = (
        positions[k]
    )

    velocity_k = (
        velocities[k]
    )

    drag_k = (
        production_quadratic_drag_force(
            velocity_k,
            drag_coefficient,
        )
    )

    net_force_k = (
        applied_force
        + drag_k
    )

    acceleration_k = (
        acceleration_from_force(
            net_force_k,
            mass,
        )
    )

    positions[k + 1] = (
        integrate_position(
            position_k,
            velocity_k,
            dt,
        )
    )

    velocities[k + 1] = (
        integrate_velocity(
            velocity_k,
            acceleration_k,
            dt,
        )
    )

    drag_forces[k] = (
        drag_k
    )

    net_forces[k] = (
        net_force_k
    )

    accelerations[k] = (
        acceleration_k
    )


# ---------------------------------------------------------------------
# Complete final-state force values
# ---------------------------------------------------------------------

drag_forces[-1] = (
    production_quadratic_drag_force(
        velocities[-1],
        drag_coefficient,
    )
)

net_forces[-1] = (
    applied_force
    + drag_forces[-1]
)

accelerations[-1] = (
    acceleration_from_force(
        net_forces[-1],
        mass,
    )
)

speeds[:] = np.linalg.norm(
    velocities,
    axis=1,
)


# ---------------------------------------------------------------------
# Analytical transient
# ---------------------------------------------------------------------

analytical_velocity_x = (
    terminal_speed_analytical
    * np.tanh(
        lambda_rate * time
    )
)


# ---------------------------------------------------------------------
# Error histories
# ---------------------------------------------------------------------

velocity_error_x = (
    velocities[:, 0]
    - analytical_velocity_x
)

terminal_speed_absolute_error = (
    np.abs(
        terminal_speed_analytical
        - speeds
    )
)


# ---------------------------------------------------------------------
# Physical and numerical validation
# ---------------------------------------------------------------------

all_finite = all([
    np.all(
        np.isfinite(
            positions
        )
    ),
    np.all(
        np.isfinite(
            velocities
        )
    ),
    np.all(
        np.isfinite(
            drag_forces
        )
    ),
    np.all(
        np.isfinite(
            net_forces
        )
    ),
    np.all(
        np.isfinite(
            accelerations
        )
    ),
    np.all(
        np.isfinite(
            speeds
        )
    ),
])


speed_monotonic = np.all(
    np.diff(
        speeds
    )
    >= -1e-12
)


no_terminal_overshoot = (
    np.max(
        speeds
    )
    <= terminal_speed_analytical
    + 1e-12
)


no_cross_axis_motion = np.allclose(
    velocities[:, 1:],
    0.0,
    atol=1e-12,
    rtol=0.0,
)


drag_power = np.einsum(
    "ij,ij->i",
    drag_forces,
    velocities,
)


drag_dissipative = np.all(
    drag_power
    <= 1e-12
)


assert all_finite

assert speed_monotonic

assert no_terminal_overshoot

assert no_cross_axis_motion

assert drag_dissipative


# ---------------------------------------------------------------------
# Confirm this really is the production drag implementation
# ---------------------------------------------------------------------

production_drag_module = (
    production_quadratic_drag_force.__module__
)

assert production_drag_module == (
    "hydrolab3d.dynamics.drag"
)


# ---------------------------------------------------------------------
# Output directories
# ---------------------------------------------------------------------

output_root = (
    repo_root
    / "results"
    / "notebooks"
    / "03_hydrodynamic_drag"
)

data_dir = (
    output_root
    / "data"
)

plots_dir = (
    output_root
    / "plots"
)


data_dir.mkdir(
    parents=True,
    exist_ok=True,
)

plots_dir.mkdir(
    parents=True,
    exist_ok=True,
)


# ---------------------------------------------------------------------
# Save canonical CSV
# ---------------------------------------------------------------------

csv_path = (
    data_dir
    / "terminal_velocity_validation.csv"
)


csv_header = ",".join([
    "time_s",
    "applied_force_x_N",
    "applied_force_y_N",
    "applied_force_z_N",
    "drag_force_x_N",
    "drag_force_y_N",
    "drag_force_z_N",
    "net_force_x_N",
    "net_force_y_N",
    "net_force_z_N",
    "acceleration_x_mps2",
    "acceleration_y_mps2",
    "acceleration_z_mps2",
    "velocity_x_mps",
    "velocity_y_mps",
    "velocity_z_mps",
    "speed_mps",
    "position_x_m",
    "position_y_m",
    "position_z_m",
    "analytical_velocity_x_mps",
    "terminal_speed_analytical_mps",
    "velocity_error_x_mps",
    "terminal_speed_absolute_error_mps",
])


csv_data = np.column_stack([
    time,
    applied_forces,
    drag_forces,
    net_forces,
    accelerations,
    velocities,
    speeds,
    positions,
    analytical_velocity_x,
    np.full_like(
        time,
        terminal_speed_analytical,
    ),
    velocity_error_x,
    terminal_speed_absolute_error,
])


np.savetxt(
    csv_path,
    csv_data,
    delimiter=",",
    header=csv_header,
    comments="",
    fmt="%.12e",
)


# ---------------------------------------------------------------------
# Figure 1: velocity validation
# ---------------------------------------------------------------------

velocity_plot_path = (
    plots_dir
    / "velocity_validation.png"
)


fig, ax = plt.subplots(
    figsize=(9, 5)
)


ax.plot(
    time,
    velocities[:, 0],
    label="Numerical velocity",
)


ax.plot(
    time,
    analytical_velocity_x,
    linestyle="--",
    label="Analytical transient",
)


ax.axhline(
    terminal_speed_analytical,
    linestyle=":",
    label="Analytical terminal speed",
)


ax.set_xlabel(
    "Time [s]"
)

ax.set_ylabel(
    "Velocity x [m/s]"
)

ax.set_title(
    "Quadratic Drag Terminal-Velocity Validation"
)

ax.grid(
    True
)

ax.legend()


fig.tight_layout()


fig.savefig(
    velocity_plot_path,
    dpi=160,
)


plt.close(
    fig
)


# ---------------------------------------------------------------------
# Figure 2: force balance
# ---------------------------------------------------------------------

force_plot_path = (
    plots_dir
    / "force_balance.png"
)


fig, ax = plt.subplots(
    figsize=(9, 5)
)


ax.plot(
    time,
    applied_forces[:, 0],
    label="Applied force",
)


ax.plot(
    time,
    drag_forces[:, 0],
    label="Drag force",
)


ax.plot(
    time,
    net_forces[:, 0],
    label="Net force",
)


ax.axhline(
    0.0,
    linewidth=1.0,
)


ax.set_xlabel(
    "Time [s]"
)

ax.set_ylabel(
    "Force x [N]"
)

ax.set_title(
    "Applied Force, Hydrodynamic Drag, and Net Force"
)

ax.grid(
    True
)

ax.legend()


fig.tight_layout()


fig.savefig(
    force_plot_path,
    dpi=160,
)


plt.close(
    fig
)


# ---------------------------------------------------------------------
# Read back and verify saved CSV
# ---------------------------------------------------------------------

saved_data = np.genfromtxt(
    csv_path,
    delimiter=",",
    names=True,
)


assert len(
    saved_data
) == (
    num_steps + 1
)


assert np.all(
    np.isfinite(
        saved_data[
            "speed_mps"
        ]
    )
)


assert np.isclose(
    saved_data[
        "velocity_x_mps"
    ][-1],
    velocities[-1, 0],
    atol=1e-10,
    rtol=0.0,
)


# ---------------------------------------------------------------------
# Verify every persistent artifact
# ---------------------------------------------------------------------

artifact_paths = [
    csv_path,
    velocity_plot_path,
    force_plot_path,
]


for artifact_path in artifact_paths:

    assert (
        artifact_path.exists()
    )

    assert (
        artifact_path.stat().st_size
        > 0
    )


# ---------------------------------------------------------------------
# Final canonical metrics
# ---------------------------------------------------------------------

final_speed = (
    speeds[-1]
)


final_absolute_speed_error = abs(
    terminal_speed_analytical
    - final_speed
)


final_relative_speed_error = (
    final_absolute_speed_error
    / terminal_speed_analytical
)


max_transient_error = np.max(
    np.abs(
        velocity_error_x
    )
)


max_transient_error_index = np.argmax(
    np.abs(
        velocity_error_x
    )
)


max_transient_error_time = (
    time[
        max_transient_error_index
    ]
)


velocity_rmse = np.sqrt(
    np.mean(
        velocity_error_x ** 2
    )
)


final_force_balance_error = abs(
    net_forces[-1, 0]
)


# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------

print(
    "Production-code canonical validation"
)

print(
    "-" * 66
)

print(
    f"Drag implementation module:   "
    f"{production_drag_module}"
)

print()

print(
    f"Analytical terminal speed:    "
    f"{terminal_speed_analytical:.12f} m/s"
)

print(
    f"Final numerical speed:        "
    f"{final_speed:.12f} m/s"
)

print(
    f"Final absolute speed error:   "
    f"{final_absolute_speed_error:.12e} m/s"
)

print(
    f"Final relative speed error:   "
    f"{final_relative_speed_error:.12e}"
)

print()

print(
    f"Maximum transient error:      "
    f"{max_transient_error:.12e} m/s"
)

print(
    f"Maximum-error time:           "
    f"{max_transient_error_time:.6f} s"
)

print(
    f"Velocity RMSE:                "
    f"{velocity_rmse:.12e} m/s"
)

print()

print(
    f"Final drag force:             "
    f"{drag_forces[-1, 0]:.12f} N"
)

print(
    f"Final net force:              "
    f"{net_forces[-1, 0]:.12e} N"
)

print(
    f"Final force-balance error:    "
    f"{final_force_balance_error:.12e} N"
)

print(
    f"Final acceleration:           "
    f"{accelerations[-1, 0]:.12e} m/s^2"
)

print()

print(
    f"Final x-position:             "
    f"{positions[-1, 0]:.12f} m"
)

print()

print(
    f"All states finite:            "
    f"{all_finite}"
)

print(
    f"Speed monotonic:              "
    f"{speed_monotonic}"
)

print(
    f"No terminal overshoot:        "
    f"{no_terminal_overshoot}"
)

print(
    f"No cross-axis motion:         "
    f"{no_cross_axis_motion}"
)

print(
    f"Drag dissipative:             "
    f"{drag_dissipative}"
)

print()

print(
    f"Saved CSV rows:               "
    f"{len(saved_data)}"
)

print()

print(
    "Persistent artifacts:"
)

for artifact_path in artifact_paths:

    print(
        " - "
        + str(
            artifact_path.relative_to(
                repo_root
            )
        )
    )


print()

print(
    "Production-code validation and "
    "artifact generation passed."
)

Production dynamics API refreshed.
Available required symbols: ['quadratic_drag_force', 'acceleration_from_force', 'integrate_position', 'integrate_velocity']

Production-code canonical validation
------------------------------------------------------------------
Drag implementation module:   hydrolab3d.dynamics.drag

Analytical terminal speed:    2.000000000000 m/s
Final numerical speed:        1.999999942053 m/s
Final absolute speed error:   5.794670432380e-08 m/s
Final relative speed error:   2.897335216190e-08

Maximum transient error:      1.104717980683e-03 m/s
Maximum-error time:           3.620000 s
Velocity RMSE:                3.943739891567e-04 m/s

Final drag force:             -11.999999304640 N
Final net force:              6.953604412274e-07 N
Final force-balance error:    6.953604412274e-07 N
Final acceleration:           3.476802206137e-08 m/s^2

Final x-position:             55.375149810591 m

All states finite:            True
Speed monotonic:              True
No te

## Step 9 - Final Repository Regression and Engineering Handoff Audit

Notebook 03 has now completed its scientific and production-code objectives.

The validated quadratic drag model has been:

1. implemented locally;
2. validated algebraically;
3. integrated with the inherited translational dynamics chain;
4. validated under passive deceleration;
5. validated against analytical terminal velocity;
6. compared against the complete analytical transient;
7. checked for first-order explicit-Euler timestep convergence;
8. promoted into the HydroLab-3D production package;
9. protected by automated regression tests;
10. exercised through the production API to generate persistent scientific artifacts.

Before closing the notebook, one final repository-level audit is required.

This step does **not** introduce new physics.

Its purpose is to confirm that Notebook 03 leaves HydroLab-3D in a clean engineering state suitable for Notebook 04.

### Final Regression Objective

The earlier production promotion test executed:

- `tests/test_drag.py`;
- `tests/test_dynamics.py`;

and reported:

`31 passed`

The final audit will now execute the complete currently available test suite:

```text
tests/
```

This checks whether introducing hydrodynamic drag accidentally affected any other already-tested HydroLab-3D capability.

### Fresh-Process Import Validation

Jupyter maintains an import cache, so the current notebook kernel is not sufficient evidence that a future Python process can cleanly import the new public API.

A fresh Python subprocess will therefore import:

```python
from hydrolab3d.dynamics import (
    quadratic_drag_force,
    acceleration_from_force,
    integrate_position,
    integrate_velocity,
)
```

This verifies the production package interface independently of the current notebook state.

### Production Source Audit

The final audit will verify that the expected production files exist and are nonempty:

- `src/hydrolab3d/dynamics/drag.py`;
- `src/hydrolab3d/dynamics/__init__.py`;
- `tests/test_drag.py`.

It will also verify that the public dynamics package source explicitly exports:

`quadratic_drag_force`

### Persistent Artifact Audit

Notebook 03 is expected to leave behind exactly the canonical scientific artifacts generated in Step 8:

- `results/notebooks/03_hydrodynamic_drag/data/terminal_velocity_validation.csv`;
- `results/notebooks/03_hydrodynamic_drag/plots/velocity_validation.png`;
- `results/notebooks/03_hydrodynamic_drag/plots/force_balance.png`.

The CSV should contain:

$$
3001
$$

samples corresponding to:

$$
T=30\ \text{s}
$$

with:

$$
\Delta t=0.01\ \text{s}
$$

including both endpoints.

The final row should preserve the validated near-terminal state:

$$
v
\approx
2\ \text{m/s}
$$

with drag approximately balancing the constant applied force.

### Engineering Handoff Criterion

Notebook 03 is ready to conclude only if:

- the entire current pytest suite passes;
- a fresh Python process can import the new production dynamics API;
- the promoted drag source and regression tests exist;
- the expected persistent artifacts exist and are nonempty;
- the canonical CSV contains the expected number of samples;
- its final numerical state is finite;
- its final speed remains consistent with the validated terminal-velocity result.

If these conditions hold, Notebook 03 has no remaining implementation work.

The notebook may then close with its final Conclusion cell and hand the validated translational force balance to Notebook 04.

In [11]:
from pathlib import Path
import os
import subprocess
import sys

import numpy as np


# ---------------------------------------------------------------------
# Resolve repository
# ---------------------------------------------------------------------

repo_root = Path.cwd().resolve()

if repo_root.name == "notebooks":
    repo_root = repo_root.parent

src_path = repo_root / "src"


# ---------------------------------------------------------------------
# Fresh-process environment for src-layout package
# ---------------------------------------------------------------------

subprocess_environment = os.environ.copy()

existing_pythonpath = subprocess_environment.get(
    "PYTHONPATH",
    "",
)

if existing_pythonpath:
    subprocess_environment["PYTHONPATH"] = (
        str(src_path)
        + os.pathsep
        + existing_pythonpath
    )
else:
    subprocess_environment["PYTHONPATH"] = str(
        src_path
    )


# ---------------------------------------------------------------------
# 1. Run COMPLETE currently available pytest suite
# ---------------------------------------------------------------------

pytest_result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests",
        "-q",
    ],
    cwd=repo_root,
    env=subprocess_environment,
    capture_output=True,
    text=True,
)


print("Complete HydroLab-3D pytest suite")
print("-" * 64)

print(pytest_result.stdout)

if pytest_result.stderr.strip():
    print("pytest stderr")
    print("-" * 64)
    print(pytest_result.stderr)


assert pytest_result.returncode == 0, (
    "Full repository regression suite failed."
)


# ---------------------------------------------------------------------
# 2. Verify public production API from a FRESH Python process
# ---------------------------------------------------------------------

fresh_import_code = """
from hydrolab3d.dynamics import (
    quadratic_drag_force,
    acceleration_from_force,
    integrate_position,
    integrate_velocity,
)

import numpy as np

drag = quadratic_drag_force(
    np.array([2.0, 0.0, 0.0]),
    3.0,
)

expected = np.array([
    -12.0,
    0.0,
    0.0,
])

assert np.allclose(
    drag,
    expected,
    atol=1e-12,
    rtol=0.0,
)

print("Fresh-process production API import passed.")
print("Known production drag:", drag)
"""


import_result = subprocess.run(
    [
        sys.executable,
        "-c",
        fresh_import_code,
    ],
    cwd=repo_root,
    env=subprocess_environment,
    capture_output=True,
    text=True,
)


print("Fresh-process API validation")
print("-" * 64)

print(import_result.stdout)

if import_result.stderr.strip():
    print("Fresh-process stderr")
    print("-" * 64)
    print(import_result.stderr)


assert import_result.returncode == 0, (
    "Fresh-process production API validation failed."
)


# ---------------------------------------------------------------------
# 3. Production source-file audit
# ---------------------------------------------------------------------

production_files = [
    repo_root
    / "src"
    / "hydrolab3d"
    / "dynamics"
    / "drag.py",

    repo_root
    / "src"
    / "hydrolab3d"
    / "dynamics"
    / "__init__.py",

    repo_root
    / "tests"
    / "test_drag.py",
]


for path in production_files:

    assert path.exists(), (
        f"Missing production file: {path}"
    )

    assert path.stat().st_size > 0, (
        f"Production file is empty: {path}"
    )


dynamics_init_text = (
    production_files[1]
    .read_text(
        encoding="utf-8"
    )
)

assert "quadratic_drag_force" in (
    dynamics_init_text
)


# ---------------------------------------------------------------------
# 4. Persistent Notebook 03 artifact audit
# ---------------------------------------------------------------------

output_root = (
    repo_root
    / "results"
    / "notebooks"
    / "03_hydrodynamic_drag"
)

csv_path = (
    output_root
    / "data"
    / "terminal_velocity_validation.csv"
)

velocity_plot_path = (
    output_root
    / "plots"
    / "velocity_validation.png"
)

force_plot_path = (
    output_root
    / "plots"
    / "force_balance.png"
)


artifact_paths = [
    csv_path,
    velocity_plot_path,
    force_plot_path,
]


for path in artifact_paths:

    assert path.exists(), (
        f"Missing artifact: {path}"
    )

    assert path.stat().st_size > 0, (
        f"Artifact is empty: {path}"
    )


# ---------------------------------------------------------------------
# 5. Validate canonical CSV contents
# ---------------------------------------------------------------------

saved_data = np.genfromtxt(
    csv_path,
    delimiter=",",
    names=True,
)


expected_sample_count = 3001

assert len(saved_data) == (
    expected_sample_count
)


required_columns = {
    "time_s",
    "applied_force_x_N",
    "drag_force_x_N",
    "net_force_x_N",
    "acceleration_x_mps2",
    "velocity_x_mps",
    "speed_mps",
    "position_x_m",
    "analytical_velocity_x_mps",
    "terminal_speed_analytical_mps",
    "velocity_error_x_mps",
    "terminal_speed_absolute_error_mps",
}


actual_columns = set(
    saved_data.dtype.names
)


missing_columns = (
    required_columns
    - actual_columns
)


assert not missing_columns, (
    "Canonical CSV missing columns: "
    + ", ".join(
        sorted(
            missing_columns
        )
    )
)


# ---------------------------------------------------------------------
# 6. Final-state validation from SAVED artifact
# ---------------------------------------------------------------------

final_time_saved = (
    saved_data["time_s"][-1]
)

final_speed_saved = (
    saved_data["speed_mps"][-1]
)

final_drag_saved = (
    saved_data["drag_force_x_N"][-1]
)

final_net_force_saved = (
    saved_data["net_force_x_N"][-1]
)

final_acceleration_saved = (
    saved_data[
        "acceleration_x_mps2"
    ][-1]
)

terminal_speed_saved = (
    saved_data[
        "terminal_speed_analytical_mps"
    ][-1]
)

final_position_saved = (
    saved_data["position_x_m"][-1]
)


final_values = np.array([
    final_time_saved,
    final_speed_saved,
    final_drag_saved,
    final_net_force_saved,
    final_acceleration_saved,
    terminal_speed_saved,
    final_position_saved,
])


assert np.all(
    np.isfinite(
        final_values
    )
)


assert np.isclose(
    final_time_saved,
    30.0,
    atol=1e-12,
    rtol=0.0,
)


assert np.isclose(
    terminal_speed_saved,
    2.0,
    atol=1e-12,
    rtol=0.0,
)


# Reuse the ACTUAL measured Notebook 03 terminal result rather than
# inventing a new arbitrary tolerance.
assert np.isclose(
    final_speed_saved,
    1.999999942053,
    atol=1e-12,
    rtol=0.0,
)


# ---------------------------------------------------------------------
# 7. Final repository handoff report
# ---------------------------------------------------------------------

print("Final Notebook 03 engineering audit")
print("-" * 64)

print(
    f"Canonical CSV samples:        "
    f"{len(saved_data)}"
)

print(
    f"Final saved time:             "
    f"{final_time_saved:.12f} s"
)

print(
    f"Final saved speed:            "
    f"{final_speed_saved:.12f} m/s"
)

print(
    f"Analytical terminal speed:    "
    f"{terminal_speed_saved:.12f} m/s"
)

print(
    f"Final saved drag force:       "
    f"{final_drag_saved:.12f} N"
)

print(
    f"Final saved net force:        "
    f"{final_net_force_saved:.12e} N"
)

print(
    f"Final saved acceleration:     "
    f"{final_acceleration_saved:.12e} m/s^2"
)

print(
    f"Final saved x-position:       "
    f"{final_position_saved:.12f} m"
)

print()

print("Production files:")

for path in production_files:

    print(
        " - "
        + str(
            path.relative_to(
                repo_root
            )
        )
    )


print()

print("Persistent artifacts:")

for path in artifact_paths:

    print(
        " - "
        + str(
            path.relative_to(
                repo_root
            )
        )
    )


print()

print(
    "Final repository regression and "
    "engineering handoff audit passed."
)

Complete HydroLab-3D pytest suite
----------------------------------------------------------------
...............................                                          [100%]
31 passed in 0.06s

Fresh-process API validation
----------------------------------------------------------------
Fresh-process production API import passed.
Known production drag: [-12.  -0.  -0.]

Final Notebook 03 engineering audit
----------------------------------------------------------------
Canonical CSV samples:        3001
Final saved time:             30.000000000000 s
Final saved speed:            1.999999942053 m/s
Analytical terminal speed:    2.000000000000 m/s
Final saved drag force:       -11.999999304640 N
Final saved net force:        6.953604412274e-07 N
Final saved acceleration:     3.476802206137e-08 m/s^2
Final saved x-position:       55.375149810590 m

Production files:
 - src/hydrolab3d/dynamics/drag.py
 - src/hydrolab3d/dynamics/__init__.py
 - tests/test_drag.py

Persistent artifacts:

# Conclusion

## What Was Implemented

Notebook 03 introduced the first velocity-dependent hydrodynamic effect into HydroLab-3D using the reduced-order quadratic drag model:

$$
\mathbf{F}_D
=
-C_D
\|\mathbf{v}\|
\mathbf{v}
$$

where $C_D$ is an effective quadratic drag coefficient for the reduced translational model.

The drag force was integrated into the existing HydroLab-3D translational dynamics chain through:

$$
\mathbf{F}_{net}
=
\mathbf{F}_{applied}
+
\mathbf{F}_D
$$

followed by:

$$
\mathbf{a}
=
\frac{\mathbf{F}_{net}}{m}
$$

and the previously validated explicit-Euler updates:

$$
\mathbf{p}_{k+1}
=
\mathbf{p}_k
+
\mathbf{v}_k\Delta t
$$

$$
\mathbf{v}_{k+1}
=
\mathbf{v}_k
+
\mathbf{a}_k\Delta t
$$

The drag implementation was first developed and validated locally inside the notebook and was only promoted into the production package after the physical and numerical behavior had been established.

---

## What Was Validated

The quadratic drag model was validated at several levels.

### Algebraic Drag Properties

The notebook confirmed that:

- zero velocity produces zero drag;
- zero drag coefficient produces zero drag;
- a known one-axis case produces the expected force;
- drag always opposes nonzero velocity;
- reversing velocity reverses drag;
- doubling speed in the same direction increases drag magnitude by a factor of four;
- arbitrary 3D velocity produces drag anti-parallel to the velocity vector;
- invalid velocity shapes are rejected;
- nonfinite velocity values are rejected;
- nonscalar drag coefficients are rejected;
- nonfinite drag coefficients are rejected;
- negative drag coefficients are rejected.

For the representative 3D case used during validation:

$$
\mathbf{v}
=
[2,-1,0.5]\ \text{m/s}
$$

the drag model produced:

$$
\mathbf{F}_D
\approx
[-7.79038,\ 3.89519,\ -1.94759]\ \text{N}
$$

with:

$$
\mathbf{F}_D\cdot\mathbf{v}
\approx
-20.44974
$$

confirming dissipative behavior.

The measured quadratic scaling ratio was:

$$
4.000000000000
$$

when velocity magnitude was doubled.

### Passive Drag Deceleration

With no externally applied force, a vehicle initialized at:

$$
v_0
=
2.0\ \text{m/s}
$$

slowed to:

$$
v
=
0.587599691718\ \text{m/s}
$$

after the passive-drag experiment.

Its kinetic energy decreased from:

$$
40.0\ \text{J}
$$

to:

$$
3.452733977074\ \text{J}
$$

while the vehicle continued moving forward to:

$$
x
=
8.158639103114\ \text{m}
$$

Speed remained monotonic, kinetic energy remained monotonic, drag remained dissipative, no velocity reversal occurred, and no cross-axis motion developed.

### Terminal Velocity Under Constant Applied Force

The canonical force-driven case used:

$$
m
=
20\ \text{kg}
$$

$$
F_A
=
12\ \text{N}
$$

$$
C_D
=
3
$$

$$
\Delta t
=
0.01\ \text{s}
$$

$$
T
=
30\ \text{s}
$$

For positive one-axis motion, equilibrium requires:

$$
F_A
=
C_Dv_\infty^2
$$

so the analytical terminal speed is:

$$
v_\infty
=
\sqrt{\frac{F_A}{C_D}}
=
2.0\ \text{m/s}
$$

The production simulation reached:

$$
v_{final}
=
1.999999942053\ \text{m/s}
$$

giving an absolute terminal-speed error of:

$$
5.794670432380\times10^{-8}\ \text{m/s}
$$

and relative error:

$$
2.897335216190\times10^{-8}
$$

The final hydrodynamic drag force was:

$$
F_D
=
-11.999999304640\ \text{N}
$$

against the constant applied force:

$$
F_A
=
12.0\ \text{N}
$$

leaving a final net force of:

$$
F_{net}
=
6.953604412274\times10^{-7}\ \text{N}
$$

and final acceleration:

$$
a
=
3.476802206137\times10^{-8}\ \text{m/s}^2
$$

This numerically demonstrates the expected force balance:

$$
F_A+F_D
\rightarrow
0
$$

as:

$$
v
\rightarrow
v_\infty
$$

No terminal-speed overshoot occurred.

### Analytical Transient Validation

For the canonical case starting from rest, the analytical velocity transient is:

$$
v(t)
=
v_\infty
\tanh
\left(
\frac{\sqrt{F_AC_D}}{m}t
\right)
$$

with transient rate:

$$
\lambda
=
\frac{\sqrt{F_AC_D}}{m}
=
0.3\ \text{s}^{-1}
$$

The explicit-Euler numerical trajectory was compared against this analytical solution over the complete simulation.

The maximum transient velocity error was:

$$
1.104717980683\times10^{-3}\ \text{m/s}
$$

occurring at:

$$
t
=
3.62\ \text{s}
$$

The velocity RMSE over the full canonical 30-second trajectory was:

$$
3.943739891567\times10^{-4}\ \text{m/s}
$$

The numerical trajectory therefore reproduces both the correct transient behavior and the correct asymptotic equilibrium.

### Timestep Convergence

A focused timestep-refinement experiment used:

$$
\Delta t
=
0.04,\ 0.02,\ 0.01,\ 0.005\ \text{s}
$$

The corresponding maximum transient velocity errors were approximately:

$$
4.43183\times10^{-3}
$$

$$
2.21161\times10^{-3}
$$

$$
1.10472\times10^{-3}
$$

$$
5.52089\times10^{-4}\ \text{m/s}
$$

Successive error ratios were:

$$
2.00389395,\quad
2.00196523,\quad
2.00097823
$$

and the measured convergence orders were:

$$
1.00280616,\quad
1.00141692,\quad
1.00070547
$$

This confirms approximately first-order convergence, consistent with the explicit-Euler integration scheme inherited from Notebook 02.

No additional numerical-method study was required.

---

## Key Results

The primary validated result of Notebook 03 is that the reduced HydroLab-3D translational dynamics now support:

$$
m\dot{\mathbf{v}}
=
\mathbf{F}_{applied}
-
C_D
\|\mathbf{v}\|
\mathbf{v}
$$

with physically consistent dissipative behavior.

For the canonical constant-force validation:

- analytical terminal speed: `2.000000000000 m/s`;
- final numerical speed: `1.999999942053 m/s`;
- final absolute terminal-speed error: `5.794670432380e-08 m/s`;
- final relative terminal-speed error: `2.897335216190e-08`;
- maximum transient velocity error: `1.104717980683e-03 m/s`;
- velocity RMSE: `3.943739891567e-04 m/s`;
- final drag force: `-11.999999304640 N`;
- final net force: `6.953604412274e-07 N`;
- final acceleration: `3.476802206137e-08 m/s²`;
- final x-position: `55.375149810590 m`;
- canonical CSV samples: `3001`.

All numerical histories remained finite.

Speed increased monotonically toward terminal velocity without overshoot.

No unintended cross-axis motion occurred.

Drag remained dissipative throughout the simulation.

---

## Production-Code Promotion

The validated drag implementation was promoted into:

`src/hydrolab3d/dynamics/drag.py`

with the production function:

`quadratic_drag_force()`

The public dynamics package interface was updated in:

`src/hydrolab3d/dynamics/__init__.py`

so that HydroLab-3D now exposes:

```python
from hydrolab3d.dynamics import (
    quadratic_drag_force,
    integrate_position,
    acceleration_from_force,
    integrate_velocity,
)
```

Focused automated regression tests were added in:

`tests/test_drag.py`

The final complete currently available HydroLab-3D test suite produced:

`31 passed`

A fresh independent Python process also successfully imported the production dynamics API and evaluated the known case:

$$
\mathbf{v}
=
[2,0,0]\ \text{m/s}
$$

$$
C_D
=
3
$$

producing:

$$
\mathbf{F}_D
=
[-12,0,0]\ \text{N}
$$

This confirms that the new drag API is available independently of the Jupyter kernel state.

---

## Generated Outputs

Notebook 03 generated the following persistent scientific artifacts:

- `results/notebooks/03_hydrodynamic_drag/data/terminal_velocity_validation.csv`
- `results/notebooks/03_hydrodynamic_drag/plots/velocity_validation.png`
- `results/notebooks/03_hydrodynamic_drag/plots/force_balance.png`

The canonical CSV contains `3001` samples covering:

$$
0
\le
t
\le
30\ \text{s}
$$

at:

$$
\Delta t
=
0.01\ \text{s}
$$

and records applied force, drag force, net force, acceleration, velocity, speed, position, analytical velocity, terminal speed, and numerical errors.

---

## Output Location

All persistent Notebook 03 artifacts are stored under:

`results/notebooks/03_hydrodynamic_drag/`

with:

```text
results/notebooks/03_hydrodynamic_drag/
├── data/
│   └── terminal_velocity_validation.csv
└── plots/
    ├── velocity_validation.png
    └── force_balance.png
```

---

## Important Observations

Quadratic drag introduces fundamentally different behavior from the constant-force system validated in Notebook 02.

With no resistance, constant applied force produces constant acceleration.

With quadratic drag:

$$
F_D
\propto
v^2
$$

so increasing velocity increases resistance, which reduces the remaining net force.

The system therefore evolves through the feedback chain:

```text
velocity increases
        ↓
drag magnitude increases
        ↓
net force decreases
        ↓
acceleration decreases
        ↓
velocity approaches equilibrium
```

Terminal velocity is therefore not an imposed velocity limit.

It emerges naturally from force balance.

The passive-drag experiment also demonstrated that the model dissipates kinetic energy rather than generating it.

The transient comparison revealed a small positive explicit-Euler velocity error during acceleration, with the largest discrepancy occurring before equilibrium. The timestep-refinement study showed that this error decreases almost exactly linearly with timestep, as expected for a first-order explicit-Euler method.

The existing Notebook 02 integration convention can therefore remain unchanged.

---

## Current Limitations

Notebook 03 intentionally uses a simplified hydrodynamic model.

The coefficient:

$$
C_D
$$

is an effective reduced-order coefficient.

It is not yet the conventional dimensionless drag coefficient used in a geometric hydrodynamic expression such as:

$$
F_D
=
\frac{1}{2}
\rho C_DAv^2
$$

The present model does not yet include:

- explicit water density;
- reference area;
- geometry-dependent drag coefficients;
- axis-dependent or anisotropic damping;
- linear damping;
- rotational damping;
- vehicle angular dynamics;
- ocean currents;
- relative water-current velocity;
- added mass;
- Coriolis or centripetal hydrodynamic terms;
- turbulence;
- CFD;
- gravity;
- buoyancy;
- thrusters;
- actuator dynamics;
- full 6-DOF marine dynamics.

The canonical validation also uses aligned one-axis motion because this provides the clearest analytical reference for terminal velocity.

The underlying production drag function itself supports arbitrary 3D translational velocity vectors.

---

## How This Result Will Be Used

Notebook 03 leaves behind a reusable production hydrodynamic resistance layer.

Future HydroLab-3D development may now construct translational net force as:

$$
\mathbf{F}_{net}
=
\mathbf{F}_{applied}
+
\mathbf{F}_D
$$

with:

$$
\mathbf{F}_D
=
-C_D
\|\mathbf{v}\|
\mathbf{v}
$$

and pass that force through the already validated production dynamics functions.

The reusable translational chain is now:

```text
Current Velocity
      ↓
quadratic_drag_force(...)
      ↓
Hydrodynamic Drag
      ↓
Applied Forces + Drag
      ↓
Net Force
      ↓
acceleration_from_force(...)
      ↓
Acceleration
      ↓
integrate_position(...)
integrate_velocity(...)
      ↓
Updated Translational State
```

Notebook 03 therefore advances HydroLab-3D from ideal force-driven translation to the first explicitly marine, velocity-dependent physical effect.

---

## Next Notebook

The next development notebook is:

`04_buoyancy_gravity.ipynb`

Notebook 04 may begin from the already validated translational force balance:

$$
m\dot{\mathbf{v}}
=
\mathbf{F}_{applied}
+
\mathbf{F}_D
$$

and introduce gravitational and buoyancy forces:

$$
\mathbf{F}_G
$$

and:

$$
\mathbf{F}_B
$$

so that the translational force balance can evolve toward:

$$
m\dot{\mathbf{v}}
=
\mathbf{F}_{applied}
+
\mathbf{F}_D
+
\mathbf{F}_G
+
\mathbf{F}_B
$$

Notebook 04 does **not** need to revalidate quadratic hydrodynamic drag.

It may treat the following as inherited validated HydroLab-3D capabilities:

- 3D position integration;
- body/world reference-frame infrastructure;
- force-to-acceleration conversion;
- explicit-Euler velocity integration;
- effective quadratic translational drag;
- drag dissipation properties;
- passive drag-induced deceleration;
- constant-force terminal-velocity behavior;
- analytical transient agreement;
- expected first-order timestep convergence;
- production drag implementation;
- automated drag regression protection.

The validated development lineage is now:

```text
Notebook 00
3D Position Propagation
      ↓
Notebook 01
Orientation and Reference Frames
      ↓
Notebook 02
Force → Acceleration → Velocity
      ↓
Notebook 03
Velocity-Dependent Hydrodynamic Drag
      ↓
Notebook 04
Gravity and Buoyancy
```

HydroLab-3D now has a validated translational resistance layer ready to support the next stage of marine vehicle physics.

---